# TA Targeted Analysis — Processing Pipeline

Processes raw targeted-analysis exports into a censored, QC-assessed dataset,
following `TA Code Spec.md`.

## This file is the template. Don't run a batch in it.

The copy in the repository root is the **template**: the code, with the batch
inputs left blank. To process a batch, copy this notebook into a folder under
`runs/` and work in that copy:

```
runs/2026-08-04-oyster/TA_Processing.ipynb
```

`runs/` is gitignored, so a batch's masses, results and figures stay on your
machine while the code stays shared. Each run is also a frozen record of how that
batch was processed — later fixes to the template do not reach back and silently
change a result you have already reported.

**Only two things are filled in per batch**, both in §4 and both marked with a
banner in the cell itself: the SOP constants, and each sample's amount and the
batch's phase. Everything else is the same for every run. `answers.yaml` holds
only the path to your data folder, because that is specific to your machine
rather than to the batch.

When you improve the code, make the change in the template on a branch and open a
pull request, as `CONTRIBUTING.md` describes — not in a run copy.

## Reading the code

`docs/superpowers/specs/2026-09-15-ta-processing-design.md` records the places
where this notebook deliberately differs from the spec, and why. Several exist
because the spec was written before it was checked against real exports, and some
of its exact strings do not appear in the data.

## Status

| Spec section | State |
|---|---|
| §1 Readfile | Implemented and verified |
| §2 RT Validation | Implemented and verified |
| §3 LOQ/ULOQ | Implemented and verified |
| §4 Extraction factor | Implemented and verified |
| §5 MDL / MRL | Implemented and verified |
| §6 Spike recovery | Implemented and verified |
| §7 EIS recovery | Implemented and verified |
| §8–§11 | Not yet written |

Sections are built and verified one at a time.

## Setup

In [146]:
from pathlib import Path

import pandas as pd
import yaml

# Show every compound when printing tables rather than an elided middle.
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 160)

# Template or run? The template is the copy sitting beside TA Code Spec.md in
# the repository root. A run is a copy under runs/, which is where batch values
# belong. Printed so it is never a surprise which one you are typing into.
IS_TEMPLATE = Path('TA Code Spec.md').exists()
if IS_TEMPLATE:
    print('This is the TEMPLATE.')
    print('Start a batch with:  python new_run.py <name>   then work in that copy.')
else:
    print(f'Run folder: {Path.cwd().name}')

This is the TEMPLATE.
Start a batch with:  python new_run.py <name>   then work in that copy.


## Sample Type vocabulary

The exports label every row with a `Sample Type`. Later sections select rows by
comparing against these values, so they are defined once here and never typed as
literal strings further down.

Two of them differ from `TA Code Spec.md`. The spec says instrument blanks are
`"Blank"` and check standards are `"Check Standard"`; the exports actually use
`"Matrix Blank"` and `"Chk Std"`. Comparing against the spec's wording would match
no rows at all — and would do so silently, producing an empty result rather than an
error. The check at the end of §1 exists to catch exactly that, now and if the
instrument software changes its wording later.

In [147]:
# Sample Type values, exactly as they appear in the exports.
CAL_STD = 'Cal Std'
UNKNOWN = 'Unknown'
INSTRUMENT_BLANK = 'Matrix Blank'   # TA Code Spec.md calls this 'Blank'
CHECK_STANDARD = 'Chk Std'          # TA Code Spec.md calls this 'Check Standard'

KNOWN_SAMPLE_TYPES = {CAL_STD, UNKNOWN, INSTRUMENT_BLANK, CHECK_STANDARD}

# Labels that appear in Calculated Amount in place of a number.
# N/F comes from the instrument; the rest are written by §3 and §5.
NOT_FOUND = 'N/F'
BELOW_LOQ = '<LOQ'
ABOVE_ULOQ = '>ULOQ'
BELOW_MDL = '<MDL'
BELOW_MRL = '<MRL'
CENSOR_LABELS = {NOT_FOUND, BELOW_LOQ, ABOVE_ULOQ, BELOW_MDL, BELOW_MRL}

# Compounds retired from the method. They still appear in older exports but are
# not part of the analysis, so they are dropped as the files are read and take no
# part in anything downstream. Add a compound here when it leaves the method.
#   diSAmPAP — removed from the method; not expected in future batches.
EXCLUDED_COMPOUNDS = {'diSAmPAP'}

## §1.1 — Data folder

**To run a different batch, change `DATA_FOLDER` in the cell below.** That one line
is the only place the folder is set.

Leave it as `None` and the notebook will ask you for the folder when you run the
cell — which is what a lab member opening this for the first time will get. Either
way the answer is remembered in `answers.yaml`, so the second pass over QC-adjusted
data does not ask again and an old run can be reproduced later.

`answers.yaml` is gitignored, because the path inside it is specific to your
machine. Setting `DATA_FOLDER` below always wins over whatever is saved there.

In [148]:
# ══════════════════════════════════════════════════════════════════════════
#  SET THE DATA FOLDER HERE  —  this is the only place it is set.
#
#  Paste the folder holding this batch's CSV exports between the quotes,
#  keeping the r before the first quote:
#
#      DATA_FOLDER = r'F:\School Folder\...\26_08_04_Oyster_RawData'
#
#  Leave it as None to be asked for the folder when you run this cell.
# ══════════════════════════════════════════════════════════════════════════

DATA_FOLDER = None

# ══════════════════════════════════════════════════════════════════════════

ANSWERS_PATH = Path('answers.yaml')

# utf-8-sig, not utf-8: Windows Notepad saves a byte-order mark by default, and
# with plain utf-8 that mark becomes part of the first key name, so every value
# in the file silently goes missing.
answers = yaml.safe_load(ANSWERS_PATH.read_text(encoding='utf-8-sig')) if ANSWERS_PATH.exists() else {}
answers = answers or {}

if DATA_FOLDER:
    answers['data_folder'] = str(DATA_FOLDER)
    print('Using the folder set above.')
elif answers.get('data_folder'):
    print('Using the folder remembered in answers.yaml.')
else:
    # Paths pasted from Explorer often arrive wrapped in quotes.
    answers['data_folder'] = input('Folder holding the CSV exports: ').strip().strip('"\'')
    print('Saved to answers.yaml.')

ANSWERS_PATH.write_text(yaml.safe_dump(answers, sort_keys=False), encoding='utf-8')

DATA_FOLDER = Path(answers['data_folder'])
if not DATA_FOLDER.is_dir():
    raise NotADirectoryError(f'Not a folder: {DATA_FOLDER}')

print(f'Data folder: {DATA_FOLDER}')

Using the folder remembered in answers.yaml.
Data folder: F:\School Folder\NYU\Coding Projects\Fluorine-Mass-Balance-Pipeline\26_08_04_Oyster_RawData


## §1.2 — Read every export into one master table

Each export file holds exactly one compound, so the master table is the
concatenation of all of them.

**Missing values.** Columns are read as text so pandas cannot pick a different
dtype per file depending on which sentinels that file happens to contain. Note that
pandas still applies its own missing-value detection while doing so: `N/A` is on its
default list and becomes `NaN` at read time, while `N/F` is not and survives as
text. That split is what we want — `N/A` in `Theoretical Amount` means the field
does not apply to that row, whereas `N/F` is an instrument result meaning the
compound was looked for and not found, which §2 and §3 must preserve.

**Stray rows.** At least one export (`NaDONA`) ends with an extra line naming no
sample and no compound, carrying two unlabelled numbers in `Total Area` and
`ISTD Area` and nothing else. It appears to be an artifact of the export rather than
a measurement. Every real result belongs to a sample, so rows with no
`Sample Raw File Name` are dropped — and reported by file rather than removed
quietly, because a rising count here would mean something changed upstream.

`source_file` is added so any row can be traced back to the file it came from.

In [149]:
FILE_PATTERN = '*Quantitation_ByCompound*.csv'

export_files = sorted(DATA_FOLDER.glob(FILE_PATTERN))
if not export_files:
    raise FileNotFoundError(f'No files matching {FILE_PATTERN} in {DATA_FOLDER}')

frames = []
stray_rows = []
for path in export_files:
    one_file = pd.read_csv(path, dtype=str)

    # A real measurement always names its sample. Anything else is an export
    # artifact, not data. Checked on the raw column before any renaming.
    before = len(one_file)
    one_file = one_file[one_file['Sample Raw File Name'].notna()]
    dropped = before - len(one_file)
    if dropped:
        stray_rows.append((path.name, dropped))

    one_file['source_file'] = path.name
    frames.append(one_file)

master = pd.concat(frames, ignore_index=True)

# Compounds no longer in the method take no part in the analysis.
retired = master['Compound Name'].isin(EXCLUDED_COMPOUNDS)

print(f'Files read: {len(export_files)}')
print(f'Rows:       {len(master) - retired.sum():,}')
print(f'Columns:    {master.shape[1]} (before selecting the ones the spec names)')

if retired.any():
    print(f'\nExcluded {retired.sum()} rows for compounds retired from the method:')
    for name in sorted(master.loc[retired, 'Compound Name'].unique()):
        print(f'  {name}')
    master = master[~retired].copy()

if stray_rows:
    print(f'\nStray rows dropped ({sum(count for _, count in stray_rows)} total):')
    for name, count in stray_rows:
        print(f'  {count} from {name}')
else:
    print('\nNo stray rows found.')

Files read: 78
Rows:       4,851
Columns:    31 (before selecting the ones the spec names)

Excluded 63 rows for compounds retired from the method:
  diSAmPAP

Stray rows dropped (1 total):
  1 from 2026_02_05_PFAS_Quantitation_ByCompound_NS_NaDONA_20260805172522.csv


## §1.3 — Keep the columns the spec names

§1.3 lists fourteen columns to record and says the rest are ignored. The exports
carry thirty.

Two of the spec's names do not match the files. `Sample Name (Batch Ordering)` is
the `Sample Name` column — the exports have a separate `Sample Order` column, and
since the spec lists `Sample ID` separately, the parenthetical is read as describing
how `Sample Name` is numbered. `ISTD Actual RT` is spelled `ISTD Actual Rt`.

Missing columns raise rather than being skipped: a renamed column upstream should
stop the run, not quietly drop data the later sections depend on.

In [150]:
SPEC_COLUMNS = [
    'Sample Raw File Name',
    'Sample Type',
    'Sample Name',          # spec: 'Sample Name (Batch Ordering)'
    'Sample ID',
    'Compound Name',
    'Detected Mass',
    'Theoretical Amount',
    'Method Apex RT',
    'Calculated Amount',
    'Peak Area',
    'ISTD Compound Name',
    'ISTD Amount',
    'ISTD Area',
    'ISTD Actual Rt',       # spec: 'ISTD Actual RT'
]

absent = [name for name in SPEC_COLUMNS if name not in master.columns]
if absent:
    raise KeyError(f'Columns named in the spec are absent from the exports: {absent}')

master = master[SPEC_COLUMNS + ['source_file']].copy()

# Trim stray whitespace so comparisons against the constants above are reliable.
for column in master.columns:
    master[column] = master[column].str.strip()

print(f'Retained {len(SPEC_COLUMNS)} spec columns plus source_file.')

Retained 14 spec columns plus source_file.


## §1.4 — Convert the numeric columns

Columns used in arithmetic are converted to numbers. Anything that cannot be parsed
— `N/F`, or `Peak index not specified` in the columns that carry it — becomes `NaN`.

**`Calculated Amount` is deliberately left as text.** §3 writes the labels `<LOQ`
and `>ULOQ` into this column, and §4.1.4 requires those labels be left in place
rather than recalculated. It therefore holds a mix of numbers and labels for the
rest of the pipeline, and every later section that does arithmetic on it must
exclude the labels explicitly and preserve them in its output.

The sentinels do **not** line up across columns, so no section may assume that a
row missing one value is missing the others. In this dataset 938 rows are `N/F` in
`Calculated Amount` but only 928 in `Method Apex RT` — ten rows have no calculated
amount yet a perfectly good retention time, which §2 has to decide what to do with.

In [151]:
NUMERIC_COLUMNS = [
    'Detected Mass',
    'Theoretical Amount',
    'Method Apex RT',
    'Peak Area',
    'ISTD Amount',
    'ISTD Area',
    'ISTD Actual Rt',
]

for column in NUMERIC_COLUMNS:
    master[column] = pd.to_numeric(master[column], errors='coerce')

# Calculated Amount has to hold both numbers and the labels <LOQ and >ULOQ,
# because §3 writes those labels into it and §4.1.4 requires them left in place.
# Reading with dtype=str gives pandas 3's string dtype, which refuses to store a
# number at all, so the column is widened to object here — before §4 multiplies
# anything by an extraction factor.
master['Calculated Amount'] = master['Calculated Amount'].astype(object)

print('Converted to numeric:')
for column in NUMERIC_COLUMNS:
    parsed = master[column].notna().sum()
    print(f'  {column:22s} {parsed:>7,} of {len(master):,} values parsed')

# Two kinds of compound, told apart by whether the compound has an internal
# standard of its own. A labelled standard (EIS/NIS) does not — it *is* the
# internal standard. Labelled standards are spiked at one fixed concentration
# rather than run as a calibration curve, so §3 gives them no LOQ and §9/§10
# exclude them. Derived once here and used wherever that split is needed.
has_own_istd = master.groupby('Compound Name')['ISTD Compound Name'].apply(lambda s: s.notna().any())
LABELLED_STANDARDS = set(has_own_istd[~has_own_istd].index)
TARGET_COMPOUNDS = set(has_own_istd[has_own_istd].index)

Converted to numeric:
  Detected Mass            3,897 of 4,851 values parsed
  Theoretical Amount       2,781 of 4,851 values parsed
  Method Apex RT           3,985 of 4,851 values parsed
  Peak Area                4,842 of 4,851 values parsed
  ISTD Amount              2,835 of 4,851 values parsed
  ISTD Area                2,833 of 4,851 values parsed
  ISTD Actual Rt           4,851 of 4,851 values parsed


## §1 — Checks

What to look at before moving on to §2:

- every row has a `Sample Type`, and every value is one the notebook recognises —
  either failure stops the run here rather than silently matching nothing in §7,
  §9 or §10
- the compound count equals the number of export files, minus any compound retired
  from the method — one compound per file, so 78 files and one exclusion gives 77
- `N/F` rows are present and still readable as `N/F`, not turned into `NaN`
- `Calculated Amount` is still text; every other numeric column is `float64`
- the count of compounds carrying no `ISTD Compound Name` matches the number of
  labelled standards in the method — those are the EIS/NIS compounds §7 and §8 need

In [152]:
print('Sample Type values observed')
print(master['Sample Type'].value_counts(dropna=False).to_string())

# Counted, not dropped: a row with no Sample Type belongs to no section of the
# spec, so it must stop the run rather than be quietly skipped.
missing_type = master['Sample Type'].isna().sum()
if missing_type:
    raise ValueError(
        f'{missing_type} row(s) have no Sample Type. '
        'Check the exports for stray or partial lines.'
    )

unrecognised = set(master['Sample Type']) - KNOWN_SAMPLE_TYPES
if unrecognised:
    raise ValueError(
        f'Unrecognised Sample Type value(s): {sorted(unrecognised)}. '
        'Add them to the vocabulary cell above and check which sections they belong to.'
    )
print('\nEvery row has a recognised Sample Type.')

print(f'\nExport files:  {len(export_files)}')
print(f'Compounds:     {master["Compound Name"].nunique()}')
print(f'Samples:       {master["Sample Raw File Name"].nunique()}')
print(f'Rows:          {len(master):,}')

not_found_rows = (master['Calculated Amount'] == NOT_FOUND).sum()
print(f'\nRows with Calculated Amount = {NOT_FOUND}: {not_found_rows:,}')

print(f'Target compounds:    {len(TARGET_COMPOUNDS)}')
print(f'Labelled standards:  {len(LABELLED_STANDARDS)}  (EIS/NIS, used in §7 and §8)')

print('\nColumn dtypes')
print(master.dtypes.to_string())

Sample Type values observed
Sample Type
Matrix Blank    2233
Unknown         1309
Cal Std          924
Chk Std          385

Every row has a recognised Sample Type.

Export files:  78
Compounds:     77
Samples:       63
Rows:          4,851

Rows with Calculated Amount = N/F: 876
Target compounds:    45
Labelled standards:  32  (EIS/NIS, used in §7 and §8)

Column dtypes
Sample Raw File Name        str
Sample Type                 str
Sample Name                 str
Sample ID                   str
Compound Name               str
Detected Mass           float64
Theoretical Amount      float64
Method Apex RT          float64
Calculated Amount        object
Peak Area               float64
ISTD Compound Name          str
ISTD Amount             float64
ISTD Area               float64
ISTD Actual Rt          float64
source_file                 str


In [153]:
# First few rows, for eyeballing that the columns line up with the exports.
master.head(10)

,Sample Raw File Name,Sample Type,Sample Name,Sample ID,Compound Name,Detected Mass,Theoretical Amount,Method Apex RT,Calculated Amount,Peak Area,ISTD Compound Name,ISTD Amount,ISTD Area,ISTD Actual Rt,source_file
0,26_08_04_Cal_1,Cal Std,3,Cal 1,10:2FTS,626.9552,9.66,8.104250,10.665,6.016562e+04,M2-10:2FTS,4830.0,37556278.0,8.1,2026_02_05_PFAS_Quantitation_ByCompound_NS_102...
1,26_08_04_Cal_2,Cal Std,4,Cal 2,10:2FTS,626.9555,19.32,8.095516,18.771,1.214667e+05,M2-10:2FTS,4830.0,36778463.0,8.1,2026_02_05_PFAS_Quantitation_ByCompound_NS_102...
2,26_08_04_Cal_3,Cal Std,5,Cal 3,10:2FTS,626.9551,48.30,8.095478,49.73,3.688425e+05,M2-10:2FTS,4830.0,37646286.0,8.1,2026_02_05_PFAS_Quantitation_ByCompound_NS_102...
3,26_08_04_Cal_4,Cal Std,6,Cal 4,10:2FTS,626.9552,96.60,8.108602,94.127,7.375160e+05,M2-10:2FTS,4830.0,38588106.0,8.1,2026_02_05_PFAS_Quantitation_ByCompound_NS_102...
4,26_08_04_Cal_5,Cal Std,7,Cal 5,10:2FTS,626.9553,193.20,8.108604,183.76,1.452902e+06,M2-10:2FTS,4830.0,38314462.0,8.1,2026_02_05_PFAS_Quantitation_ByCompound_NS_102...
5,26_08_04_Cal_6,Cal Std,8,Cal 6,10:2FTS,626.9552,483.00,8.099864,474.267,3.503388e+06,M2-10:2FTS,4830.0,35424346.0,8.1,2026_02_05_PFAS_Quantitation_ByCompound_NS_102...
6,26_08_04_Cal_7,Cal Std,9,Cal 7,10:2FTS,626.9551,966.00,8.099836,987.913,7.848667e+06,M2-10:2FTS,4830.0,37955482.0,8.1,2026_02_05_PFAS_Quantitation_ByCompound_NS_102...
7,26_08_04_Cal_8,Cal Std,10,Cal 8,10:2FTS,626.9553,1932.00,8.095492,1850.304,1.461613e+07,M2-10:2FTS,4830.0,37656960.0,8.1,2026_02_05_PFAS_Quantitation_ByCompound_NS_102...
8,26_08_04_Cal_9,Cal Std,11,Cal 9,10:2FTS,626.9551,4830.00,8.099870,4829.351,3.610022e+07,M2-10:2FTS,4830.0,35508896.0,8.1,2026_02_05_PFAS_Quantitation_ByCompound_NS_102...
9,26_08_04_Cal_10,Cal Std,12,Cal 10,10:2FTS,626.9548,9660.00,8.095506,9697.504,7.198030e+07,M2-10:2FTS,4830.0,35102647.0,8.1,2026_02_05_PFAS_Quantitation_ByCompound_NS_102...


## §2 — Retention time validation

A compound should come off the column at the same time in every sample as it does
in its calibration standards. §2.1 requires every non-standard row to sit within
±0.4 minutes of the mean retention time of that compound's `Cal Std` rows. A peak
outside that window is not that compound, so the row is removed. The calibration
standards themselves define the reference, so they are not checked against it.

**`N/F` rows are removed here too, including calibration standards.** When the
instrument reports `N/F` there is no measurement — no retention time, no amount —
so the row carries nothing to validate and nothing to carry forward. A few of these
rows do have a valid retention time, but that does not rescue them: without a
calculated amount there is no result.

Removing `N/F` calibration standards has a consequence worth understanding. §3 sets
each compound's LOQ from its *lowest surviving* calibration level, so a compound
whose lowest standards were never detected gets a higher LOQ. That is the intended
behaviour — quantitation cannot be claimed at a level the instrument could not see.
The check below lists every compound this affects.

This section also creates the **compound table**, the spec's "compound list" — one
row per compound, starting with its reference retention time. Later sections add
their own columns: LOQ and ULOQ in §3, MDL or MRL in §5, spike recoveries in §6.

Removed rows are kept in `removed_rows` rather than discarded, because the QC report
in §11 has to be able to show what was dropped and why.

In [154]:
RT_WINDOW_MIN = 0.4

# The compound table — the spec's "compound list". Later sections add columns.
compound_table = pd.DataFrame(index=sorted(master['Compound Name'].unique()))
compound_table.index.name = 'Compound Name'

# Reference retention time: the mean across each compound's calibration standards.
# N/F standards have no retention time, so they drop out of the mean on their own.
cal_std = master[master['Sample Type'] == CAL_STD]
compound_table['reference_rt'] = cal_std.groupby('Compound Name')['Method Apex RT'].mean()

no_reference = compound_table.index[compound_table['reference_rt'].isna()].tolist()
if no_reference:
    print(f'No Cal Std rows, so no reference RT, for: {no_reference}')

# How far each row sits from its own compound's reference.
reference_rt = master['Compound Name'].map(compound_table['reference_rt'])
rt_offset = (master['Method Apex RT'] - reference_rt).abs()

is_cal_std = master['Sample Type'] == CAL_STD
is_not_found = master['Calculated Amount'] == NOT_FOUND

# Two separate rules, deliberately not combined:
#   N/F means no measurement exists, so the row goes whatever its sample type —
#     including calibration standards, which is what raises the LOQ in §3 for a
#     compound whose lowest standards were never detected.
#   The RT window is checked on non-standards only, since the standards define it.
drop_rt = ~is_cal_std & ~is_not_found & (rt_offset > RT_WINDOW_MIN)
drop_row = is_not_found | drop_rt

removed_rows = master[drop_row].copy()
removed_rows['rt_offset'] = rt_offset[drop_row]
removed_rows['removed_because'] = f'RT outside +/-{RT_WINDOW_MIN} min'
removed_rows.loc[is_not_found[drop_row], 'removed_because'] = 'N/F, no measurement'

rows_before = len(master)
master = master[~drop_row].copy()

print(f'Rows before §2: {rows_before:,}')
print(removed_rows['removed_because'].value_counts().to_string())
print(f'Rows after §2:  {len(master):,}')

Rows before §2: 4,851
removed_because
N/F, no measurement      876
RT outside +/-0.4 min      1
Rows after §2:  3,974


## §2 — Checks

What to look at before moving on to §3:

- the rows removed plus the rows kept add back up to the rows we started with
- no `N/F` values survive in `Calculated Amount`
- no surviving non-standard row sits more than 0.4 minutes from its reference
- every compound has a reference retention time
- the compounds losing the most rows are ones you would expect to be near the
  detection limit, not something surprising — a compound losing almost everything
  is worth investigating before trusting the rest of the run

In [155]:
assert len(master) + len(removed_rows) == rows_before, 'rows went missing'
assert not (master['Calculated Amount'] == NOT_FOUND).any(), 'N/F survived §2'
assert compound_table['reference_rt'].notna().all(), 'a compound has no reference RT'

surviving_offset = (master['Method Apex RT'] - master['Compound Name'].map(compound_table['reference_rt'])).abs()
worst = surviving_offset[master['Sample Type'] != CAL_STD].max()
assert worst <= RT_WINDOW_MIN, f'a row survived at {worst:.3f} min from reference'
print(f'All checks passed. Widest surviving offset: {worst:.3f} min (limit {RT_WINDOW_MIN}).')

print('\nRemoved by reason and sample type')
print(removed_rows.groupby(['removed_because', 'Sample Type']).size().to_string())

# Calibration standards lost to N/F matter more than other losses: they set the
# LOQ and ULOQ in §3, so losing the low end of a curve raises that compound's LOQ.
lost_cal = removed_rows[removed_rows['Sample Type'] == CAL_STD]
if len(lost_cal):
    kept_cal = master[master['Sample Type'] == CAL_STD]
    print(f'\nCalibration levels lost ({len(lost_cal)} rows) — these raise LOQ in §3')
    for name, grp in lost_cal.groupby('Compound Name'):
        remaining = kept_cal.loc[kept_cal['Compound Name'] == name, 'Theoretical Amount']
        print(f'  {name:12s} lost {len(grp):2d} levels, {len(remaining):2d} remain,'
              f' lowest now {remaining.min():g}')

print('\nCompounds losing the most rows')
summary = pd.DataFrame({
    'removed': removed_rows.groupby('Compound Name').size(),
    'kept': master.groupby('Compound Name').size(),
}).fillna(0).astype(int)
print(summary.sort_values('removed', ascending=False).head(10).to_string())

All checks passed. Widest surviving offset: 0.264 min (limit 0.4).

Removed by reason and sample type
removed_because        Sample Type 
N/F, no measurement    Cal Std          10
                       Matrix Blank    543
                       Unknown         323
RT outside +/-0.4 min  Matrix Blank      1

Calibration levels lost (10 rows) — these raise LOQ in §3
  FHEA         lost  3 levels,  9 remain, lowest now 100
  FOEA         lost  1 levels, 11 remain, lowest now 20
  FprPA        lost  1 levels, 11 remain, lowest now 20
  HFPO-DA      lost  4 levels,  8 remain, lowest now 200
  N-MeFOSAA    lost  1 levels, 11 remain, lowest now 20

Compounds losing the most rows
               removed  kept
Compound Name               
HFPO-DA             44    19
FHEA                40    23
N-MeFOSA            39    24
FpePA               37    26
N-MeFOSAA           36    27
11Cl-PF3OUdS        35    28
6:2FTS              34    29
N-EtFOSA            34    29
FOEA                33    3

## §3 — LOQ / ULOQ and censoring

A calibration curve only supports quantitation between its lowest and highest
standards. §3.1 sets each compound's **LOQ** to the lowest theoretical amount among
its calibration standards and its **ULOQ** to the highest, then censors any result
falling outside that range: below becomes `<LOQ`, above becomes `>ULOQ`.

Only standards that survived §2 count, so a level the instrument reported as `N/F`
cannot become the LOQ. This is where the five raised LOQs from §2 take effect, and
the check below shows how many extra results each one censors.

**Limits apply to target compounds only.** A labelled standard is spiked into every
sample at one fixed concentration rather than run as a curve, so its lowest and
highest calibration levels are the same number and an LOQ would be meaningless.
Those 32 compounds get no limits and are never censored; §7 and §8 use their peak
areas directly, and §9 and §10 exclude them explicitly.

**Censoring applies to unknowns and blanks, not to standards** — a departure from
§3.3's "every row in the master list", made deliberately. Censoring exists to stop
a result outside the quantifiable range being reported as a number. A standard is
not a reported result; it is the evidence establishing where that range lies.
Censoring them did two kinds of damage when tried:

- Endpoint standards scatter across their own limit by measurement noise. PFOA's
  top standard read 50002.554 against a 50000 theoretical — 100.005% recovery, an
  excellent point — and was labelled `>ULOQ`. Every one of the 18 affected
  calibration standards sat at an endpoint of its own curve.
- It destroyed §10's input. HFPO-DA's `Check Cal 5` read 166 against a theoretical
  200 — 83%, a pass under §10's 70–130% window — but HFPO-DA's LOQ had been raised
  to 200 by §2, so the value was overwritten with `<LOQ` and §10 could no longer
  assess it.

Standards keeping their numbers costs nothing: a standard that genuinely failed is
still visible, and the check below reports any outside 70–130% of its own
theoretical value.

**The label replaces the number in `Calculated Amount`**, as §3.3 and §3.4 require.
That column therefore holds a mix of numbers and labels from here on. Every later
section that does arithmetic on it — §4's EF, §5's MDL averaging, §6's spike
recovery — has to exclude the labels first and preserve them in its output.

In [156]:
# LOQ and ULOQ apply to target compounds only, from the calibration levels that
# survived §2. Labelled standards are spiked at a single fixed concentration, so
# they have no curve and get no limits — they keep NaN here.
target_cal = master[master['Compound Name'].isin(TARGET_COMPOUNDS) & (master['Sample Type'] == CAL_STD)]
compound_table['loq'] = target_cal.groupby('Compound Name')['Theoretical Amount'].min()
compound_table['uloq'] = target_cal.groupby('Compound Name')['Theoretical Amount'].max()
compound_table['cal_levels_used'] = target_cal.groupby('Compound Name').size()

# Censoring applies to reported results, not to the standards that establish the
# range. Standards keep their numbers so §10 can compare check standards against
# cal standards, and so an endpoint standard scattering a fraction of a percent
# past its own theoretical value is not mislabelled as out of range.
CENSORED_SAMPLE_TYPES = {UNKNOWN, INSTRUMENT_BLANK}

# Calculated Amount holds text so the labels can live in it, so compare on a
# numeric copy and write the labels back into the text column. A labelled
# standard's limits are NaN, and every comparison against NaN is False, so those
# rows are never censored.
amount = pd.to_numeric(master['Calculated Amount'], errors='coerce')
loq = master['Compound Name'].map(compound_table['loq'])
uloq = master['Compound Name'].map(compound_table['uloq'])

censorable = master['Sample Type'].isin(CENSORED_SAMPLE_TYPES)
below_loq = censorable & (amount < loq)
above_uloq = censorable & (amount > uloq)

master.loc[below_loq, 'Calculated Amount'] = BELOW_LOQ
master.loc[above_uloq, 'Calculated Amount'] = ABOVE_ULOQ

print(f'Censored {below_loq.sum():,} results as {BELOW_LOQ}')
print(f'Censored {above_uloq.sum():,} results as {ABOVE_ULOQ}')
print(f'Quantifiable target results in censorable samples: {(censorable & ~below_loq & ~above_uloq & loq.notna()).sum():,}')
print(f'Standards left uncensored: {(~censorable).sum():,}')

Censored 744 results as <LOQ
Censored 1 results as >ULOQ
Quantifiable target results in censorable samples: 460
Standards left uncensored: 1,299


## §3 — Checks

What to look at before moving on to §4:

- every compound has an LOQ and a ULOQ, and the LOQ is below the ULOQ
- no surviving number sits outside its compound's range — if one did, censoring
  missed it
- the cost of the raised LOQs from §2, shown per compound: how many results were
  censored only because the lower calibration levels were lost. These would have
  been reportable numbers had those standards been detected
- censoring broken down by sample type. Unknowns and blanks below LOQ are normal.
  **A calibration or check standard falling outside its own compound's range is
  not** — it means a standard did not read back at its own concentration, which is
  worth investigating before trusting that compound's results

In [157]:
targets = compound_table.loc[sorted(TARGET_COMPOUNDS)]
assert targets['loq'].notna().all(), 'a target compound has no LOQ'
assert (targets['loq'] < targets['uloq']).all(), 'a target compound has LOQ >= ULOQ'
assert compound_table.loc[sorted(LABELLED_STANDARDS), 'loq'].isna().all(), 'a labelled standard was given an LOQ'

still_numeric = pd.to_numeric(master['Calculated Amount'], errors='coerce')
assert not (censorable & (still_numeric < loq)).any(), 'a value below LOQ escaped censoring'
assert not (censorable & (still_numeric > uloq)).any(), 'a value above ULOQ escaped censoring'
assert still_numeric[~censorable].notna().all(), 'a standard lost its number'
print(f'All checks passed. {len(targets)} target compounds have limits, '
      f'{len(LABELLED_STANDARDS)} labelled standards correctly have none.')

print('\nCensoring by sample type (target compounds only)')
target_rows = master[master['Compound Name'].isin(TARGET_COMPOUNDS)]
label = target_rows['Calculated Amount'].where(target_rows['Calculated Amount'].isin(CENSOR_LABELS), 'quantified')
print(pd.crosstab(target_rows['Sample Type'], label).to_string())

# Standards are not censored, so an out-of-range one is reported here instead —
# a standard that did not read back at its own concentration is a QC concern.
std = target_rows[~target_rows['Sample Type'].isin(CENSORED_SAMPLE_TYPES)].copy()
std['recovery_pct'] = 100 * pd.to_numeric(std['Calculated Amount'], errors='coerce') / std['Theoretical Amount']
poor = std[(std['recovery_pct'] < 70) | (std['recovery_pct'] > 130)]
print(f'\nStandards outside 70-130% of their own theoretical value: {len(poor)}')
if len(poor):
    print(poor[['Compound Name', 'Sample Type', 'Sample ID', 'Theoretical Amount', 'recovery_pct']]
          .sort_values('recovery_pct').to_string(index=False))

# What the raised LOQs from §2 actually cost, using the levels lost there.
lost_cal = removed_rows[removed_rows['Sample Type'] == CAL_STD]
if len(lost_cal):
    print('\nResults censored only because §2 removed the lower calibration levels')
    for name, was in lost_cal.groupby('Compound Name')['Theoretical Amount'].min().items():
        now = compound_table.loc[name, 'loq']
        only_because = (censorable & (master['Compound Name'] == name) & (amount >= was) & (amount < now)).sum()
        print(f'  {name:12s} LOQ {was:g} -> {now:g},  {only_because} result(s) lost')

All checks passed. 45 target compounds have limits, 32 labelled standards correctly have none.

Censoring by sample type (target compounds only)
Calculated Amount  <LOQ  >ULOQ  quantified
Sample Type                               
Cal Std               0      0         530
Chk Std               0      0         225
Matrix Blank        648      0         113
Unknown              96      1         347

Standards outside 70-130% of their own theoretical value: 10
Compound Name Sample Type   Sample ID  Theoretical Amount  recovery_pct
      HFPO-DA     Chk Std Check Cal 4              100.00     21.111000
         FDEA     Chk Std Check Cal 4              100.00     60.435000
      HFPO-DA     Cal Std       Cal 5              200.00     61.934500
       NaDONA     Cal Std       Cal 1                9.45    135.216931
     N-EtFOSA     Cal Std       Cal 1               10.00    136.450000
      HFPO-DA     Chk Std Check Cal 8             2000.00    138.499900
       PFTrDA     Cal Std      

## §3 — QC deliverable: LOQ and ULOQ per compound

The LOQ and ULOQ are reportable QC values in their own right, not just internal
thresholds, so they are printed here as a table and kept in `compound_table` for
the QC Report in §11.

`compound_table` is where every per-compound QC value accumulates as the pipeline
runs: the reference retention time from §2, the limits below, MDL or MRL from §5,
spike recoveries from §6. §11 reports it rather than recalculating anything.

`cal_levels_used` is included because an LOQ means something different when it
rests on 8 calibration levels than on 12 — it records how much curve is actually
behind each limit.

In [158]:
loq_report = compound_table.loc[sorted(TARGET_COMPOUNDS), ['loq', 'uloq', 'cal_levels_used']]
loq_report = loq_report.rename(columns={'loq': 'LOQ', 'uloq': 'ULOQ', 'cal_levels_used': 'Cal levels'})

print(f'LOQ / ULOQ per target compound ({len(loq_report)} compounds)')
print(loq_report.sort_index().to_string())

reduced = loq_report[loq_report['Cal levels'] < loq_report['Cal levels'].max()]
if len(reduced):
    print(f'\n{len(reduced)} compound(s) rest on fewer than the full '
          f'{loq_report["Cal levels"].max()} calibration levels:')
    print(reduced.sort_values('Cal levels').to_string())

LOQ / ULOQ per target compound (45 compounds)
                  LOQ     ULOQ  Cal levels
Compound Name                             
10:2FTS          9.66  48300.0        12.0
11Cl-PF3OUdS     9.43  47150.0        12.0
4:2FTS           9.37  46850.0        12.0
6:2FTS           9.51  47550.0        12.0
8:2FTS           9.60  48000.0        12.0
9Cl-PF3ONS       9.33  46650.0        12.0
Br-PFHxS         1.73   8650.0        12.0
Br-PFOS          1.96   9800.0        12.0
FBSA            10.00  50000.0        12.0
FDEA            10.00  50000.0        12.0
FDSA-I          10.00  50000.0        12.0
FHEA           100.00  50000.0         9.0
FOEA            20.00  50000.0        11.0
FOSA            10.00  50000.0        12.0
FhpPA           10.00  50000.0        12.0
FhxSA           10.00  50000.0        12.0
FpePA           10.00  50000.0        12.0
FpeSA-I         10.00  50000.0        12.0
FprPA           20.00  50000.0        11.0
HFPO-DA        200.00  50000.0         8.0
L-PFBS  

## §4 — Extraction factor

The instrument reports a concentration in the vial, in **ng per litre**. The
extraction factor converts that back to a concentration in the original sample:

    EF = (dilution final / dilution aliquot) x (blowdown volume in litres / amount)
       = (300 µL / 135 µL) x (0.0005 L / amount)

The first ratio reverses the dilution applied when prepping for LC-MS injection.
The second scales the blowdown volume against how much sample was actually
extracted. A smaller sample gives a larger EF, because the same measured
concentration came from less material.

**The blowdown volume has to be converted from millilitres to litres**, because
the instrument's concentration is per litre. Multiplying ng/L by a volume in
litres gives nanograms of analyte in the extract; dividing that by the sample
gives ng/g for a solid or ng/L for a liquid. Leaving the volume in millilitres
would overstate every result by a factor of 1000.

**The three constants are inputs, not fixed values.** They come from the SOP and
can change between batches, so they are entered once per batch and recorded. The
arithmetic is exactly the formula above either way.

**Each batch is one phase**, declared once, and that decides the unit its results
carry: a solid is weighed in grams and reports **ng/g**, a liquid is measured in
litres and reports **ng/L**. The arithmetic is identical; only the unit differs.
Every reported value that depends on it — final concentrations, MDL and MRL,
spike recoveries — carries the batch's unit.

**Amounts are needed only for `Unknown` samples**, per §4.1.1 — those are the ones
being quantified. Calibration standards, check standards and matrix blanks are not
converted to a per-gram or per-litre basis and keep their vial concentrations.

A sample with nothing to weigh, such as an equipment blank, is marked `auto` and
takes the smallest amount in the batch. That gives it the largest extraction
factor, so anything it contains is scaled up the most — the conservative choice,
since these blanks feed the MDL in §5.

**Censored results keep their labels**, per §4.1.4: `<LOQ` and `>ULOQ` describe a
range the EF does not shift, so the EF is applied only to rows holding a number.

In [159]:
# ══════════════════════════════════════════════════════════════════════════
#  EF CONSTANTS FOR THIS BATCH — from your SOP. Change them here.
#
#      EF = (DILUTION_FINAL_UL / DILUTION_ALIQUOT_UL) * (blowdown in L / amount)
#
#  The volume ratio reverses the dilution done when prepping for LC-MS
#  injection. BLOWDOWN_ML is the theoretical blowdown volume.
# ══════════════════════════════════════════════════════════════════════════

DILUTION_FINAL_UL = 300.0     # SOP value 300
DILUTION_ALIQUOT_UL = 135.0   # SOP value 135
BLOWDOWN_ML = 0.5             # SOP value 0.5

# ══════════════════════════════════════════════════════════════════════════

# The instrument reports a concentration in ng per LITRE, while the blowdown
# volume is in millilitres, so the volume is converted before it meets the
# concentration. Without this the results come out 1000x too high.
#
#   ng/L x litres            = ng of analyte in the extract
#   ng     / grams of sample = ng/g        (a solid batch)
#   ng     / litres of sample = ng/L       (a liquid batch)
ML_PER_L = 1000.0
BLOWDOWN_L = BLOWDOWN_ML / ML_PER_L

EF_NUMERATOR = (DILUTION_FINAL_UL / DILUTION_ALIQUOT_UL) * BLOWDOWN_L

print(f'EF = ({DILUTION_FINAL_UL:g}/{DILUTION_ALIQUOT_UL:g}) x ({BLOWDOWN_ML:g} mL as {BLOWDOWN_L:g} L / amount)')
print(f'   = {EF_NUMERATOR:.8g} / amount')

EF = (300/135) x (0.5 mL as 0.0005 L / amount)
   = 0.0011111111 / amount


In [163]:
# So that PHASE = solid works as well as PHASE = 'solid'. Quoting is a Python
# rule that has nothing to teach you about chemistry, so both spellings are fine.
solid, liquid = 'solid', 'liquid'
auto = AUTO = 'auto'

# ══════════════════════════════════════════════════════════════════════════
#  FILL IN FOR THIS BATCH — type the values here, in this cell.
#
#  PHASE           solid   -> amounts in grams,  results in ng/g
#                  liquid  -> amounts in litres, results in ng/L
#                  A batch is all one phase.
#
#  SAMPLE_AMOUNTS  one line per Unknown sample. Use auto for a sample with
#                  nothing to weigh, such as an equipment blank — it takes
#                  the smallest amount in the batch.
#
#  Don't know the sample names? Leave SAMPLE_AMOUNTS empty and run this
#  cell. It prints a ready-made block for this batch to paste back here.
# ══════════════════════════════════════════════════════════════════════════

PHASE = solid

SAMPLE_AMOUNTS = {
    '26_08_04_Oyster_TA_01': 0.59,    # S Oyster 1
    '26_08_04_Oyster_TA_02': 0.56,    # S Oyster 2
    '26_08_04_Oyster_TA_03': 0.50,    # S Oyster 3
    '26_08_04_Oyster_TA_04': 0.59,    # M Oyster 1
    '26_08_04_Oyster_TA_05': 0.50,    # M Oyster 2
    '26_08_04_Oyster_TA_06': 0.66,    # M Oyster 3
    '26_08_04_Oyster_TA_07': 0.52,    # L Oyster 1
    '26_08_04_Oyster_TA_08': 0.75,    # L Oyster 2
    '26_08_04_Oyster_TA_09': 0.67,    # L Oyster 3
    '26_08_04_Oyster_TA_10': 0.43,    # M Oyster Low Spike
    '26_08_04_Oyster_TA_11': 0.43,    # M Oyster High Spike
    '26_08_04_Oyster_TA_12': 0.41,    # Chicken Blank 1
    '26_08_04_Oyster_TA_13': 0.46,    # Chicken Blank 2
    '26_08_04_Oyster_TA_14': 0.58,    # Chicken Blank 3
    '26_08_04_Oyster_TA_15': 0.59,    # Chicken Low Spike
    '26_08_04_Oyster_TA_16': 0.53,    # Chicken High Spike
    '26_08_04_Oyster_TA_17': auto,    # Equipment Blank

}

# ══════════════════════════════════════════════════════════════════════════

PHASE_UNITS = {'solid': 'ng/g', 'liquid': 'ng/L'}

unknown_samples = sorted(master.loc[master['Sample Type'] == UNKNOWN, 'Sample Raw File Name'].unique())
sample_id = master.drop_duplicates('Sample Raw File Name').set_index('Sample Raw File Name')['Sample ID']

if not SAMPLE_AMOUNTS:
    print('Copy the block below into SAMPLE_AMOUNTS above, then fill in each amount.\n')
    print('SAMPLE_AMOUNTS = {')
    for name in unknown_samples:
        label = str(sample_id.get(name, ''))
        default = AUTO if 'equipment blank' in label.lower() else 'None'
        print(f"    '{name}': {default},{'':{max(0, 6 - len(default))}}  # {label}")
    print('}')
    raise ValueError('SAMPLE_AMOUNTS is empty — paste the block printed above into this cell.')

phase = str(PHASE or '').strip().lower()
problems = [] if phase in PHASE_UNITS else [('PHASE', f'must be solid or liquid, not {PHASE!r}')]
problems += [(n, 'missing from SAMPLE_AMOUNTS') for n in unknown_samples if n not in SAMPLE_AMOUNTS]
problems += [(n, 'not a sample in this batch') for n in SAMPLE_AMOUNTS if n not in unknown_samples]
for name in unknown_samples:
    value = SAMPLE_AMOUNTS.get(name)
    if value is None:
        problems.append((name, f'needs an amount, or {AUTO}'))
    elif str(value).strip().lower() != AUTO and not float(value) > 0:
        problems.append((name, f'amount must be greater than zero, not {value!r}'))

if problems:
    print(f'{len(problems)} entr(ies) need attention:\n')
    for name, why in problems:
        print(f'  {name:26s} {str(sample_id.get(name, "")):22s} {why}')
    raise ValueError('Fix the entries listed above in this cell, then re-run it.')

BATCH_UNITS = PHASE_UNITS[phase]
print(f'Batch phase: {phase}  ->  results reported in {BATCH_UNITS}')

Batch phase: solid  ->  results reported in ng/g


In [164]:
# The sample table — the spec's "sample list", the per-sample counterpart to
# compound_table. Later sections add spike assignments and matrix groupings.
sample_table = pd.DataFrame(index=unknown_samples)
sample_table.index.name = 'Sample Raw File Name'
sample_table['sample_id'] = sample_id.reindex(unknown_samples)

weighed = {name: float(value) for name, value in SAMPLE_AMOUNTS.items()
           if str(value).strip().lower() != AUTO}

# An equipment blank carries no sample of its own. Giving it the smallest amount
# in the batch gives it the largest EF, so whatever it contains is scaled up the
# most — the conservative choice, since these blanks feed the MDL in §5.
derived_amount = min(weighed.values())
sample_table['amount'] = [weighed.get(name, derived_amount) for name in unknown_samples]
sample_table['amount_source'] = ['weighed' if name in weighed else f'auto ({derived_amount:g})'
                                 for name in unknown_samples]
sample_table['ef'] = EF_NUMERATOR / sample_table['amount']
MAX_EF = sample_table['ef'].max()

amount_now = pd.to_numeric(master['Calculated Amount'], errors='coerce')
has_number = amount_now.notna()

# Each Unknown sample is converted by its own EF. §4.1.4: censored rows keep
# their label, because <LOQ and >ULOQ describe a range the EF does not shift.
own_ef = master['Sample Raw File Name'].map(sample_table['ef'])
apply_ef = (master['Sample Type'] == UNKNOWN) & has_number

# An instrument blank carries no sample either, so it has no EF of its own. It
# is converted by the batch's largest EF so that §5 can compare it against the
# detection limits on the same basis. The largest EF is the most conservative
# choice: it scales any contamination up the most, so nothing slips past. This
# is the same factor the equipment blank receives, since the largest EF is by
# definition the one belonging to the smallest amount.
apply_max_ef = (master['Sample Type'] == INSTRUMENT_BLANK) & has_number

master.loc[apply_ef, 'Calculated Amount'] = amount_now[apply_ef] * own_ef[apply_ef]
master.loc[apply_max_ef, 'Calculated Amount'] = amount_now[apply_max_ef] * MAX_EF

print(f'EF applied to {apply_ef.sum():,} results across {len(sample_table)} samples.')
print(f'Largest EF ({MAX_EF:.6g}) applied to {apply_max_ef.sum():,} instrument blank results.')
print(f'Censored rows left untouched: {(has_number.eq(False) & master["Sample Type"].isin([UNKNOWN, INSTRUMENT_BLANK])).sum():,}')
print(f'\nEF range across samples: {sample_table["ef"].min():.6g} to {MAX_EF:.6g}')

EF applied to 889 results across 17 samples.
Largest EF (0.00271003) applied to 1,041 instrument blank results.
Censored rows left untouched: 745

EF range across samples: 0.00148148 to 0.00271003


## §4 — Checks

What to look at before moving on to §5:

- every `Unknown` sample has an amount and an EF, and no EF is zero or infinite
- the derived amount used for the equipment blank is the smallest weighed amount
  in the batch, and the table says plainly which samples were weighed and which
  were derived
- censored rows still read `<LOQ` or `>ULOQ` — the EF must not have turned a label
  into a number, nor a number into a label
- results grew by the EF and nothing else. The check re-derives one sample's
  values from the pre-EF numbers and compares, so an EF applied twice, applied to
  the wrong sample, or silently skipped would show up here
- nothing outside `Unknown` changed: standards and matrix blanks keep their vial
  concentrations

In [165]:
assert sample_table['ef'].notna().all() and (sample_table['ef'] > 0).all(), 'a sample has a bad EF'
assert derived_amount == min(weighed.values()), 'derived amount is not the smallest weighed amount'
assert MAX_EF == EF_NUMERATOR / min(sample_table['amount']), 'largest EF is not the smallest amount''s EF'

after = pd.to_numeric(master['Calculated Amount'], errors='coerce')

# Each sample converted by its own factor; each instrument blank by the largest.
assert ((after[apply_ef] - amount_now[apply_ef] * own_ef[apply_ef]).abs() < 1e-12).all(), \
    'a sample result is not its pre-EF value times its own EF'
assert ((after[apply_max_ef] - amount_now[apply_max_ef] * MAX_EF).abs() < 1e-12).all(), \
    'an instrument blank is not its pre-EF value times the largest EF'

# Labels survived untouched, and no label became a number.
assert (master['Calculated Amount'].isin(CENSOR_LABELS) == amount_now.isna()).all(), \
    'censoring labels changed during §4'

# Standards keep their vial concentrations; only samples and blanks convert.
standards = master['Sample Type'].isin([CAL_STD, CHECK_STANDARD])
assert ((after[standards] - amount_now[standards]).abs().fillna(0) < 1e-12).all(), 'a standard changed in §4'
print(f'All checks passed. Samples and instrument blanks now in {BATCH_UNITS}; standards unchanged.')

print(f'\nSample table ({len(sample_table)} Unknown samples)')
print(sample_table[['sample_id', 'amount', 'amount_source', 'ef']].to_string())

print('\nCensoring unchanged by §4')
converted = master[master['Sample Type'].isin([UNKNOWN, INSTRUMENT_BLANK])]
print(converted['Calculated Amount'].where(
    converted['Calculated Amount'].isin(CENSOR_LABELS), 'quantified').value_counts().to_string())

All checks passed. Samples and instrument blanks now in ng/g; standards unchanged.

Sample table (17 Unknown samples)
                                 sample_id  amount amount_source        ef
Sample Raw File Name                                                      
26_08_04_Oyster_TA_01           S Oyster 1    0.59       weighed  0.001883
26_08_04_Oyster_TA_02           S Oyster 2    0.56       weighed  0.001984
26_08_04_Oyster_TA_03           S Oyster 3    0.50       weighed  0.002222
26_08_04_Oyster_TA_04           M Oyster 1    0.59       weighed  0.001883
26_08_04_Oyster_TA_05           M Oyster 2    0.50       weighed  0.002222
26_08_04_Oyster_TA_06           M Oyster 3    0.66       weighed  0.001684
26_08_04_Oyster_TA_07           L Oyster 1    0.52       weighed  0.002137
26_08_04_Oyster_TA_08           L Oyster 2    0.75       weighed  0.001481
26_08_04_Oyster_TA_09           L Oyster 3    0.67       weighed  0.001658
26_08_04_Oyster_TA_10   M Oyster Low Spike    0.43       

## §5 — MDL and MRL

Each compound gets a detection limit, below which a result is not distinguishable
from what the method itself contributes.

**MDL** is measured from the method blanks: `mean + t(n-1, 0.99) × standard
deviation` of the compound's values across them. It needs the compound actually
detected in at least three blanks, because a standard deviation from fewer than
three numbers describes nothing.

**MRL** is the fallback when there aren't three detections: `LOQ × mean EF` across
the sample list. It is not measured from this batch's blanks at all — it is the
quantitation limit carried onto a per-sample basis.

**Expect MRL to be the usual outcome.** A batch with three method blanks needs the
compound found in *all three* to earn an MDL, and §2 removed `N/F` rows, so a blank
where the compound was not found is absent rather than present as a zero — it
cannot count toward the three. MRL is therefore the formula behind most of your
reported limits, and the one worth checking hardest.

Both carry the batch's unit from §4: ng/g for a solid batch, ng/L for a liquid one.
That holds because the blank values have already had their EF applied, and because
multiplying the LOQ by the mean EF converts it from a vial concentration to the
same per-sample basis.

The method blanks are chosen by you below — which samples served that role is not
something the data records.

In [167]:
# A standard deviation needs at least three numbers to mean anything, so a
# compound detected in fewer than three method blanks gets an MRL instead.
MIN_BLANKS_FOR_MDL = 3

# ══════════════════════════════════════════════════════════════════════════
#  METHOD BLANKS FOR THIS BATCH
#
#  List the samples that served as method blanks. Which samples played that
#  role is not recorded in the data, so it has to be stated here.
#
#  Leave the list empty and run this cell to see the batch's samples and a
#  ready-made block to paste back in.
# ══════════════════════════════════════════════════════════════════════════

METHOD_BLANKS = ['26_08_04_Oyster_TA_12',    # Chicken Blank 1
    '26_08_04_Oyster_TA_13',    # Chicken Blank 2
    '26_08_04_Oyster_TA_14',    # Chicken Blank 3
]

# ══════════════════════════════════════════════════════════════════════════

if not METHOD_BLANKS:
    print('Copy the block below into METHOD_BLANKS above, keeping only the blanks.\n')
    print('METHOD_BLANKS = [')
    for name in unknown_samples:
        print(f"    '{name}',    # {sample_id.get(name, '')}")
    print(']')
    raise ValueError('METHOD_BLANKS is empty — paste the block above and keep only the method blanks.')

not_a_sample = [name for name in METHOD_BLANKS if name not in unknown_samples]
if not_a_sample:
    raise ValueError(f'Not Unknown samples in this batch: {not_a_sample}')

print(f'{len(METHOD_BLANKS)} method blank(s):')
for name in METHOD_BLANKS:
    print(f'  {name}    {sample_id.get(name, "")}')

if len(METHOD_BLANKS) < MIN_BLANKS_FOR_MDL:
    print(f'\nFewer than {MIN_BLANKS_FOR_MDL} blanks, so every compound will fall to MRL.')

3 method blank(s):
  26_08_04_Oyster_TA_12    Chicken Blank 1
  26_08_04_Oyster_TA_13    Chicken Blank 2
  26_08_04_Oyster_TA_14    Chicken Blank 3


In [168]:
from scipy import stats

blank_rows = master[master['Sample Raw File Name'].isin(METHOD_BLANKS)]
blank_values = pd.to_numeric(blank_rows['Calculated Amount'], errors='coerce')

# §5.1.3: the MRL uses the mean EF across the whole sample list.
mean_ef = sample_table['ef'].mean()

kinds, values, counts = {}, {}, {}
for compound in sorted(TARGET_COMPOUNDS):
    detected = blank_values[blank_rows['Compound Name'] == compound].dropna()
    counts[compound] = len(detected)
    if len(detected) >= MIN_BLANKS_FOR_MDL:
        t = float(stats.t.ppf(0.99, len(detected) - 1))
        kinds[compound] = 'MDL'
        values[compound] = detected.mean() + t * detected.std(ddof=1)
    else:
        kinds[compound] = 'MRL'
        values[compound] = compound_table.loc[compound, 'loq'] * mean_ef

compound_table['limit_type'] = pd.Series(kinds)
compound_table['limit'] = pd.Series(values)
compound_table['blanks_detected'] = pd.Series(counts)

print(f'Method blanks: {len(METHOD_BLANKS)}    mean EF across the sample list: {mean_ef:.4f}')
print(compound_table['limit_type'].value_counts().to_string())

print(f'\nDetection limits ({BATCH_UNITS})')
report = compound_table.loc[sorted(TARGET_COMPOUNDS),
                            ['limit_type', 'limit', 'blanks_detected', 'loq']]
print(report.sort_values(['limit_type', 'limit']).to_string())

Method blanks: 3    mean EF across the sample list: 0.0021
limit_type
MRL    41
MDL     4

Detection limits (ng/g)
              limit_type     limit  blanks_detected     loq
Compound Name                                              
PFNA                 MDL  0.142002              3.0   10.00
L-PFHxS              MDL  0.223061              3.0    7.41
PFHxS                MDL  0.258406              3.0   10.00
PFBA                 MDL  2.139107              3.0   10.00
Br-PFHxS             MRL  0.003669              1.0    1.73
Br-PFOS              MRL  0.004157              0.0    1.96
L-PFOS               MRL  0.015524              0.0    7.32
L-PFBS               MRL  0.018812              0.0    8.87
9Cl-PF3ONS           MRL  0.019787              0.0    9.33
4:2FTS               MRL  0.019872              0.0    9.37
L-PFPeS              MRL  0.019957              0.0    9.41
11Cl-PF3OUdS         MRL  0.019999              0.0    9.43
NaDONA               MRL  0.020042           

In [169]:
# Censor results below their compound's limit. §5.1.5 says such a result is
# removed; it is labelled in place instead, as §3 does, so the final table still
# has an entry for every compound in every sample saying why there is no number.
# The label names which limit applied, because <MRL is a quantitation limit
# carried over while <MDL was measured from this batch's own blanks.
#
# Samples and instrument blanks both qualify, because §4 put both on this
# batch's per-sample basis — each sample by its own EF, each instrument blank by
# the largest EF in the batch. Calibration and check standards keep their vial
# concentrations and are not compared against a per-sample limit.
limit = master['Compound Name'].map(compound_table['limit'])
limit_label = master['Compound Name'].map(
    compound_table['limit_type'].map({'MDL': BELOW_MDL, 'MRL': BELOW_MRL}))

amount_before_mdl = pd.to_numeric(master['Calculated Amount'], errors='coerce')
below_limit = master['Sample Type'].isin(CENSORED_SAMPLE_TYPES) & (amount_before_mdl < limit)

master.loc[below_limit, 'Calculated Amount'] = limit_label[below_limit]

print(f'Censored {below_limit.sum():,} results below their limit:')
print(master.loc[below_limit, 'Calculated Amount'].value_counts().to_string())

# Counted over target compounds only. Labelled standards are never censored and
# are not reported as results, so including them would overstate this badly.
surviving = pd.to_numeric(master['Calculated Amount'], errors='coerce')
is_target_unknown = (master['Sample Type'] == UNKNOWN) & master['Compound Name'].isin(TARGET_COMPOUNDS)
print(f'\nReportable target results in Unknown samples: '
      f'{(is_target_unknown & surviving.notna()).sum():,} of {is_target_unknown.sum():,}')

# Anything left in an instrument blank after censoring is contamination above
# the detection limit, which is what §9 assesses.
ib_left = (master['Sample Type'] == INSTRUMENT_BLANK) & surviving.notna() & master['Compound Name'].isin(TARGET_COMPOUNDS)
print(f'Instrument blank results still above their limit: {ib_left.sum():,}  (assessed in §9)')

Censored 45 results below their limit:
Calculated Amount
<MDL    38
<MRL     7

Reportable target results in Unknown samples: 309 of 444
Instrument blank results still above their limit: 106  (assessed in §9)


In [170]:
assert compound_table.loc[sorted(TARGET_COMPOUNDS), 'limit'].notna().all(), 'a target compound has no limit'
assert compound_table.loc[sorted(LABELLED_STANDARDS), 'limit'].isna().all(), 'a labelled standard was given a limit'

still = pd.to_numeric(master['Calculated Amount'], errors='coerce')
censorable = master['Sample Type'].isin(CENSORED_SAMPLE_TYPES)
assert not (censorable & (still < limit)).any(), 'a result below its limit escaped censoring'

# Standards keep their vial concentrations, so §5 must leave them alone.
assert ((still[~censorable] - amount_before_mdl[~censorable]).abs().fillna(0) < 1e-12).all(), \
    'a standard changed in §5'

# An MDL rests on the blanks; an MRL does not. Every MDL compound must have had
# at least MIN_BLANKS_FOR_MDL detections, and no MRL compound may have had that many.
mdl_rows = compound_table['limit_type'] == 'MDL'
mrl_rows = compound_table['limit_type'] == 'MRL'
assert (compound_table.loc[mdl_rows, 'blanks_detected'] >= MIN_BLANKS_FOR_MDL).all(), 'an MDL rests on too few blanks'
assert (compound_table.loc[mrl_rows, 'blanks_detected'] < MIN_BLANKS_FOR_MDL).all(), 'an MRL was used despite enough blanks'
print('All checks passed.')

print(f'\nWhere every target result ended up ({BATCH_UNITS} for samples and instrument blanks)')
target_rows = master[master['Compound Name'].isin(TARGET_COMPOUNDS)]
outcome = target_rows['Calculated Amount'].where(target_rows['Calculated Amount'].isin(CENSOR_LABELS), 'quantified')
print(pd.crosstab(target_rows['Sample Type'], outcome).to_string())

# A compound that earned an MDL did so by being present in the blanks. Showing
# what its limit would have been as an MRL says how much that contamination cost.
mdl_table = compound_table.loc[mdl_rows, ['limit', 'blanks_detected', 'loq']].copy()
mdl_table['mrl_would_have_been'] = compound_table.loc[mdl_rows, 'loq'] * mean_ef
mdl_table['times_higher'] = mdl_table['limit'] / mdl_table['mrl_would_have_been']
print('\nCompounds whose limit came from the blanks rather than the LOQ')
print(mdl_table.sort_values('times_higher', ascending=False).to_string())

in_samples = master['Sample Type'] == UNKNOWN
print(f'\nSample results lost to blank contamination ({len(mdl_table)} compounds): '
      f'{(in_samples & (master["Calculated Amount"] == BELOW_MDL)).sum()}')
print(f'Sample results lost to the carried-over limit ({int(mrl_rows.sum())} compounds): '
      f'{(in_samples & (master["Calculated Amount"] == BELOW_MRL)).sum()}')

All checks passed.

Where every target result ended up (ng/g for samples and instrument blanks)
Calculated Amount  <LOQ  <MDL  <MRL  >ULOQ  quantified
Sample Type                                           
Cal Std               0     0     0      0         530
Chk Std               0     0     0      0         225
Matrix Blank        648     7     0      0         106
Unknown              96    31     7      1         309

Compounds whose limit came from the blanks rather than the LOQ
                  limit  blanks_detected    loq  mrl_would_have_been  times_higher
Compound Name                                                                     
PFBA           2.139107              3.0  10.00             0.021208    100.862060
L-PFHxS        0.223061              3.0   7.41             0.015715     14.193870
PFHxS          0.258406              3.0  10.00             0.021208     12.184237
PFNA           0.142002              3.0  10.00             0.021208      6.695613

Sample resu

## §6 — Spike recovery

A spike recovery asks a simple question: if we add a known amount of a compound
to a real sample, how much of it do we find again? A recovery near 100% says the
method extracts that compound from that matrix properly. A low one says the
matrix is holding onto it, or that it is lost during extraction.

Two spikes are run per matrix, a **low** and a **high**, because recovery is not
always the same at the bottom and the top of the working range.

    nominal concentration = (spike amount in ng / sample amount) x spike factor

    recovery % = (value in the spiked sample - background mean) / nominal x 100

**The background is subtracted** because the spiked sample already contained
whatever the matrix naturally holds. Without subtracting it, a contaminated
matrix would look like excellent recovery.

**The spike factor** corrects for the fact that a spiking solution is made up as
a salt or as a mixture of isomers, so not all of the mass added is the compound
being measured. For most compounds it is 1. It is well below 1 for the branched
isomers `Br-PFHxS` and `Br-PFOS`, and for the linear sulfonates, so those would
read far too low without it.

**Each matrix gets its own background.** The oyster spikes are compared against
unspiked oysters, the chicken spikes against the chicken blanks. Comparing a
spike against a background from a different matrix would be meaningless.

**Non-detects in the background are averaged over what was actually detected.**
Most background results are censored by §5, so the mean is taken over the
uncensored values only, and a compound detected in none of the background
samples is treated as having no measurable background. The table reports how
many background samples each mean rests on, so a recovery resting on thin
background evidence is visible rather than hidden.

A spiked result that is itself censored gets no recovery number — you cannot
compute a percentage from `<MDL`. Those are counted and listed rather than
dropped silently.

**Recovery must land between 70% and 130%.** Anything outside that window is
flagged for you to look at: too low means the compound is being lost, too high
means something is adding to it. A compound with no recovery at all is flagged
too, since an unmeasurable spike is itself a QC finding rather than a blank in
the table.

In [ ]:
# The QC window a SPIKE recovery has to fall inside. From the method, not from
# this batch. §7 has its own, wider window for EIS recovery — the two are not
# the same measurement. The flag wording is built from these numbers so it
# cannot go stale if the window is changed.
RECOVERY_MIN_PCT, RECOVERY_MAX_PCT = 70.0, 130.0

FLAG_LOW_RECOVERY = f'recovery below {RECOVERY_MIN_PCT:g}%'
FLAG_HIGH_RECOVERY = f'recovery above {RECOVERY_MAX_PCT:g}%'
FLAG_NO_RECOVERY = 'no recovery, spiked result censored'

# ══════════════════════════════════════════════════════════════════════════
#  SPIKE FACTORS — from the method, not from this batch. Change only if the
#  spiking solution changes.
#
#  The fraction of the spiked mass that is actually the compound measured.
#  1 means the whole spike counts. The six names carrying a comment are the
#  fluorotelomer carboxylic acids, which the exports name differently from the
#  way the method writes them; the comment gives the method's name.
# ══════════════════════════════════════════════════════════════════════════

SPIKE_FACTORS = {
    '10:2FTS': 0.966,   '11Cl-PF3OUdS': 0.943,  '4:2FTS': 0.937,
    '6:2FTS': 0.951,    '8:2FTS': 0.960,        '9Cl-PF3ONS': 0.933,
    'Br-PFHxS': 0.173,  'Br-PFOS': 0.196,       'NaDONA': 0.945,
    'L-PFBS': 0.887,    'L-PFDS': 0.965,        'L-PFHpS': 0.953,
    'L-PFHxS': 0.741,   'L-PFNS': 0.962,        'L-PFOS': 0.732,
    'L-PFPeS': 0.941,
    'FDEA': 1,      # 10:2 FTCA
    'FHEA': 1,      # 6:2 FTCA
    'FOEA': 1,      # 8:2 FTCA
    'FprPA': 1,     # 3:3 FTCA
    'FpePA': 1,     # 5:3 FTCA
    'FhpPA': 1,     # 7:3 FTCA
    'FBSA': 1,      'FDSA-I': 1,     'FOSA': 1,       'FhxSA': 1,
    'FpeSA-I': 1,   'HFPO-DA': 1,    'N-EtFOSA': 1,   'N-EtFOSAA': 1,
    'N-MeFOSA': 1,  'N-MeFOSAA': 1,  'PFBA': 1,       'PFDA': 1,
    'PFDoA': 1,     'PFHpA': 1,      'PFHxA': 1,      'PFHxS': 1,
    'PFNA': 1,      'PFOA': 1,       'PFOS': 1,       'PFPeA': 1,
    'PFTeDA': 1,    'PFTrDA': 1,     'PFUdA': 1,
}

# ══════════════════════════════════════════════════════════════════════════

# A compound named here that the instrument does not report, or a target
# compound with no factor, is a naming mismatch — the kind that otherwise
# silently drops compounds out of the recovery table. Stop instead.
missing_factor = sorted(TARGET_COMPOUNDS - set(SPIKE_FACTORS))
unknown_factor = sorted(set(SPIKE_FACTORS) - TARGET_COMPOUNDS)
if missing_factor or unknown_factor:
    raise KeyError(
        f'Target compounds with no spike factor: {missing_factor}\n'
        f'Spike factors naming no compound in this batch: {unknown_factor}'
    )

compound_table['spike_factor'] = pd.Series(SPIKE_FACTORS)

corrected = compound_table.loc[sorted(TARGET_COMPOUNDS), 'spike_factor']
print(f'Spike factors set for all {len(corrected)} target compounds.')
print(f'{(corrected < 1).sum()} carry a factor below 1:')
print(corrected[corrected < 1].sort_values().to_string())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  SPIKES FOR THIS BATCH — type the values here, in this cell.
#
#  LOW_SPIKE_NG    how much was spiked into the low spike sample, in NANOGRAMS
#  HIGH_SPIKE_NG   the same for the high spike sample
#
#  SPIKE_SETS      one entry per matrix. For each matrix, name the low spike
#                  sample, the high spike sample, and the unspiked samples of
#                  that same matrix that give it a background.
#
#  Don't know the sample names? Leave SPIKE_SETS empty ({}) and run this cell.
#  It prints the batch's samples and a ready-made block to paste back here.
# ══════════════════════════════════════════════════════════════════════════

LOW_SPIKE_NG = 1.1
HIGH_SPIKE_NG = 12.0

SPIKE_SETS = {
    'Oyster': {
        'low':  '26_08_04_Oyster_TA_10',            # M Oyster Low Spike
        'high': '26_08_04_Oyster_TA_11',            # M Oyster High Spike
        'background': ['26_08_04_Oyster_TA_04',     # M Oyster 1
                       '26_08_04_Oyster_TA_05',     # M Oyster 2
                       '26_08_04_Oyster_TA_06'],    # M Oyster 3
    },
    'Chicken': {
        'low':  '26_08_04_Oyster_TA_15',            # Chicken Low Spike
        'high': '26_08_04_Oyster_TA_16',            # Chicken High Spike
        'background': ['26_08_04_Oyster_TA_12',     # Chicken Blank 1
                       '26_08_04_Oyster_TA_13',     # Chicken Blank 2
                       '26_08_04_Oyster_TA_14'],    # Chicken Blank 3
    },
}

# ══════════════════════════════════════════════════════════════════════════

SPIKE_NG = {'low': LOW_SPIKE_NG, 'high': HIGH_SPIKE_NG}

if not SPIKE_SETS:
    print('Copy the block below into SPIKE_SETS above and edit it to match.\n')
    print("SPIKE_SETS = {\n    'Matrix name': {")
    print("        'low':  None,   # <- the low spike sample")
    print("        'high': None,   # <- the high spike sample")
    print("        'background': [],")
    print('    },\n}\n')
    print("This batch's samples:")
    for name in unknown_samples:
        print(f"    '{name}',    # {sample_id.get(name, '')}")
    raise ValueError('SPIKE_SETS is empty — paste the block above and fill it in.')

problems = []
for amount_name, amount in (('LOW_SPIKE_NG', LOW_SPIKE_NG), ('HIGH_SPIKE_NG', HIGH_SPIKE_NG)):
    if not (amount and float(amount) > 0):
        problems.append((amount_name, 'must be an amount in ng greater than zero'))

for matrix, chosen in SPIKE_SETS.items():
    named = [chosen.get('low'), chosen.get('high')] + list(chosen.get('background') or [])
    problems += [(matrix, f'{n!r} is not an Unknown sample in this batch')
                 for n in named if n not in unknown_samples]
    if not chosen.get('background'):
        problems.append((matrix, 'needs at least one background sample'))
    overlap = set(chosen.get('background') or []) & {chosen.get('low'), chosen.get('high')}
    if overlap:
        problems.append((matrix, f'spiked sample also listed as background: {sorted(overlap)}'))

if problems:
    print(f'{len(problems)} entr(ies) need attention:\n')
    for where, why in problems:
        print(f'  {where:12s} {why}')
    raise ValueError('Fix the entries listed above in this cell, then re-run it.')

print(f'Spike amounts: low {LOW_SPIKE_NG:g} ng, high {HIGH_SPIKE_NG:g} ng\n')
for matrix, chosen in SPIKE_SETS.items():
    print(f'{matrix}')
    for level in ('low', 'high'):
        name = chosen[level]
        print(f'  {level:4s} spike  {name}  {sample_id.get(name, "")}'
              f'   {sample_table.loc[name, "amount"]:g}')
    for name in chosen['background']:
        print(f'  background  {name}  {sample_id.get(name, "")}')

In [ ]:
# One number per sample per compound, censored results left out as NaN, so a
# mean over a background takes in only what was actually detected.
numeric = master.assign(value=pd.to_numeric(master['Calculated Amount'], errors='coerce'))
wide = numeric.pivot_table(index='Sample Raw File Name', columns='Compound Name', values='value')
wide = wide.reindex(columns=sorted(TARGET_COMPOUNDS))

spike_factor = compound_table.loc[sorted(TARGET_COMPOUNDS), 'spike_factor']

rows = []
for matrix, chosen in SPIKE_SETS.items():
    background = wide.reindex(chosen['background'])
    background_mean = background.mean()      # over detected values only
    for level in ('low', 'high'):
        sample = chosen[level]
        # ng spiked in, over how much sample it went into, times the fraction of
        # that mass which is really the compound: the concentration we expect.
        nominal = (SPIKE_NG[level] / sample_table.loc[sample, 'amount']) * spike_factor
        spiked = wide.reindex([sample]).iloc[0]
        rows.append(pd.DataFrame({
            'matrix': matrix,
            'level': level,
            'spike_sample': sample,
            'nominal': nominal,
            'background_mean': background_mean.fillna(0.0),
            'backgrounds_detected': background.count(),
            'spiked_value': spiked,
            'recovery_pct': 100 * (spiked - background_mean.fillna(0.0)) / nominal,
        }))

# One row per compound per matrix per level. The spec keeps spike results as
# columns on the compound list, which assumes a single spike pair; this batch
# has a pair per matrix, so they live in their own table instead.
spike_table = pd.concat(rows).rename_axis('Compound Name')

# A recovery outside the window is a QC finding, so it is labelled here rather
# than only counted. An unflagged row is one that passed.
recovery = spike_table['recovery_pct']
spike_table['flag'] = ''
spike_table.loc[recovery < RECOVERY_MIN_PCT, 'flag'] = FLAG_LOW_RECOVERY
spike_table.loc[recovery > RECOVERY_MAX_PCT, 'flag'] = FLAG_HIGH_RECOVERY
spike_table.loc[recovery.isna(), 'flag'] = FLAG_NO_RECOVERY

In [ ]:
computed = spike_table['recovery_pct'].notna()
print(f'Spike recoveries computed: {computed.sum():,} of {len(spike_table):,} '
      f'({len(SPIKE_SETS)} matrices x 2 levels x {len(spike_factor)} compounds)')
print('\nRecovery % by matrix and level')
print(spike_table[computed].groupby(['matrix', 'level'])['recovery_pct'].describe()[
    ['count', 'min', '50%', 'max']].to_string())
print(f'\nQC flags ({RECOVERY_MIN_PCT:g}-{RECOVERY_MAX_PCT:g}% window)')
print(f'  passed  {(spike_table["flag"] == "").sum():,} of {len(spike_table):,}')
print(spike_table.loc[spike_table['flag'] != '', 'flag'].value_counts().to_string())

## §6 — Checks

What to look at before moving on to §7:

- every matrix produced a low and a high result for every target compound, and
  the nominal concentration is positive everywhere
- the nominal concentration is the spike amount over the spiked sample's own
  amount, times that compound's spike factor. The check re-derives one by hand
- the high spike's nominal concentration is larger than the low spike's for
  every compound, since more was added
- `Br-PFHxS` and `Br-PFOS` carry the smallest spike factors, so their nominal
  concentrations should be far below their neighbours'. If they are not, the
  factors did not reach the calculation
- recoveries cluster around 100%. A compound far outside that is a real result
  worth reading, not necessarily an error — but a whole matrix far outside
  points at the spike amount or the sample amount being wrong
- a compound detected in no background sample shows `0` background and a
  `backgrounds_detected` of 0, and its recovery therefore rests on the spiked
  value alone
- the flags agree with the window: nothing between 70% and 130% carries a flag,
  and everything outside it does. The check asserts both directions
- read the flag list itself. A compound flagged in one matrix but not the other
  points at that matrix; a compound flagged at the low spike but not the high
  points at the bottom of the range, where the spike is small enough that the
  background subtraction matters most

In [ ]:
assert len(spike_table) == 2 * len(SPIKE_SETS) * len(TARGET_COMPOUNDS), 'a matrix or level is missing'
assert (spike_table['nominal'] > 0).all(), 'a nominal concentration is zero or negative'
assert spike_table['background_mean'].notna().all(), 'a background mean went missing'

# Re-derive one nominal concentration by hand, from the inputs as typed.
probe_matrix = next(iter(SPIKE_SETS))
probe_sample = SPIKE_SETS[probe_matrix]['high']
by_hand = (HIGH_SPIKE_NG / sample_table.loc[probe_sample, 'amount']) * SPIKE_FACTORS['PFOS']
in_table = spike_table.query("matrix == @probe_matrix and level == 'high'").loc['PFOS', 'nominal']
assert abs(by_hand - in_table) < 1e-12, f'nominal disagrees by hand: {by_hand} vs {in_table}'

# More spiked in must mean a higher nominal concentration, compound by compound.
for matrix in SPIKE_SETS:
    low = spike_table.query("matrix == @matrix and level == 'low'")['nominal']
    high = spike_table.query("matrix == @matrix and level == 'high'")['nominal']
    assert (high > low).all(), f'{matrix}: a high spike is not above its low spike'

# The background mean must never include a censored result.
for matrix, chosen in SPIKE_SETS.items():
    counted = spike_table.query("matrix == @matrix and level == 'low'")['backgrounds_detected']
    assert (counted <= len(chosen['background'])).all(), f'{matrix}: more backgrounds than samples'

# Every flag agrees with the window, and every unflagged row is inside it.
passed = spike_table['flag'] == ''
assert passed.eq(spike_table['recovery_pct'].between(RECOVERY_MIN_PCT, RECOVERY_MAX_PCT)).all(),     'a flag disagrees with the recovery window'
assert (spike_table.loc[spike_table['flag'] == FLAG_NO_RECOVERY, 'recovery_pct'].isna()).all(),     'a row flagged as having no recovery has one'
print('All checks passed.')

flat = spike_table.reset_index()

print(f'\nNominal spike concentrations ({BATCH_UNITS}), lowest spike factors first')
nominal_view = flat.pivot_table(index='Compound Name', columns=['matrix', 'level'], values='nominal')
print(nominal_view.loc[spike_factor.sort_values().index[:6]].to_string())

print(f'\nSpike recovery % by compound')
recovery_view = flat.pivot_table(index='Compound Name', columns=['matrix', 'level'], values='recovery_pct')
print(recovery_view.round(1).to_string())

print('\nHow many background samples each mean rests on')
print(spike_table.query("level == 'low'").pivot_table(
    index='backgrounds_detected', columns='matrix', values='nominal', aggfunc='count').to_string())

flagged = spike_table[spike_table['flag'] != ''].sort_values('recovery_pct')
print(f'\nQC FLAGS — spike recovery outside {RECOVERY_MIN_PCT:g}-{RECOVERY_MAX_PCT:g}%: '
      f'{len(flagged)} of {len(spike_table)}')
print(flagged[['matrix', 'level', 'background_mean', 'backgrounds_detected',
               'spiked_value', 'nominal', 'recovery_pct', 'flag']].round(3).to_string())

print('\nFlagged compounds by how many of their 4 spikes failed')
per_compound = flagged.groupby(level=0).size().sort_values(ascending=False)
print(per_compound.to_string())
print(f'\nCompounds passing in every matrix and level: '
      f'{len(TARGET_COMPOUNDS) - len(per_compound)} of {len(TARGET_COMPOUNDS)}')

## §7 — EIS recovery, Method 1

Labelled standards are added to every sample **before** extraction. They are the
same compounds as the targets but built with heavier atoms, so the instrument tells
them apart. Because they travel through the whole extraction with the sample, how
much comes back out measures how well the extraction worked for that sample.

Method 1 compares the EIS peak area in a sample against the same EIS in the
calibration standards and instrument blanks, which were never extracted:

    recovery % = EIS area in the sample / mean EIS area in the standards x 100

A sample at 100% recovered all of its EIS. A sample at 40% lost most of it, and
every target result in that sample is suspect for the same reason.

**EIS and NIS are not the same thing.** EIS goes in before extraction. NIS goes in
afterwards, just before injection, so it never sees the extraction and cannot
report on it. A compound is one or the other, never both — so the NIS compounds
named below are removed from the EIS list rather than being recovered against.

**Matrices.** Recovery is reported per matrix, because extraction behaves
differently in oyster than in chicken. You name the matrices and say which samples
belong to each, so a future batch of soils from several locations is just a
different set of names. A sample belonging to no matrix, such as an equipment
blank, is reported on its own.

Method 2, which uses the NIS, is a separate section and applies to only some of
these compounds.

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
#  NIS COMPOUNDS — from the method, not from this batch.
#
#  Non-extracted internal standards, added after extraction and just before
#  injection. A compound named here is NOT an EIS and is taken out of the EIS
#  list. Several NIS names differ from their EIS partner by two characters
#  (M3PFBA against MPFBA, MPFHxS against M3PFHxS, MPFOS against M8PFOS), so the
#  checks below refuse to guess if a name does not match the exports exactly.
# ═════════════════════════════════════════════════════════════════════════

# The QC window an EIS RECOVERY has to fall inside — wider than §6's spike
# window. An EIS is carried through the entire extraction, so it absorbs more
# variation than a spike measured against a known amount of matrix. Noel's
# decision, 2026-09-23: TA Code Spec.md §7.6 states 70-130%, which is the spike
# window rather than the EIS one. Both Method 1 and Method 2 use this pair.
EIS_RECOVERY_MIN_PCT, EIS_RECOVERY_MAX_PCT = 50.0, 150.0

FLAG_LOW_EIS = f'EIS recovery below {EIS_RECOVERY_MIN_PCT:g}%'
FLAG_HIGH_EIS = f'EIS recovery above {EIS_RECOVERY_MAX_PCT:g}%'
FLAG_NO_EIS = 'no EIS recovery, peak area missing'

NIS_COMPOUNDS = {
    'M3PFBA', 'MPFHxA', 'MPFOA', 'MPFNA', 'MPFDA', 'MPFHxS', 'MPFOS',
}

# ═════════════════════════════════════════════════════════════════════════

not_labelled = sorted(NIS_COMPOUNDS - LABELLED_STANDARDS)
if not_labelled:
    raise KeyError(f'Named as NIS but not a labelled standard in this batch: {not_labelled}. '
                   f'Check the spelling against the export file names.')

# §1 split every compound into targets and labelled standards by whether it has
# an internal standard of its own. Take the NIS out of the labelled standards and
# what is left is the EIS set — the compounds Method 1 recovers.
EIS_COMPOUNDS = LABELLED_STANDARDS - NIS_COMPOUNDS

# That set must equal the compounds the exports actually use as an internal
# standard. A disagreement means a compound is misfiled or the export changed.
used_as_istd = set(master['ISTD Compound Name'].dropna().unique())
if EIS_COMPOUNDS != used_as_istd:
    raise KeyError(f'EIS list disagrees with the exports. '
                   f'In the list but never used as an ISTD: {sorted(EIS_COMPOUNDS - used_as_istd)}. '
                   f'Used as an ISTD but not in the list: {sorted(used_as_istd - EIS_COMPOUNDS)}.')

print(f'EIS QC window: {EIS_RECOVERY_MIN_PCT:g}-{EIS_RECOVERY_MAX_PCT:g}% (§6 spike recovery keeps {RECOVERY_MIN_PCT:g}-{RECOVERY_MAX_PCT:g}%)')
print(f'\n{len(LABELLED_STANDARDS)} labelled standards = '
      f'{len(EIS_COMPOUNDS)} EIS + {len(NIS_COMPOUNDS)} NIS, all accounted for.')
print(f'\nNIS (added after extraction, no Method 1 recovery):')
print('  ' + ', '.join(sorted(NIS_COMPOUNDS)))

In [ ]:
# ═════════════════════════════════════════════════════════════════════════
#  MATRICES FOR THIS BATCH — type the values here, in this cell.
#
#  MATRICES     one entry per matrix, naming the samples that belong to it.
#               Extraction behaves differently in each matrix, so recovery is
#               reported per matrix. Name them to suit the batch — 'Oyster' and
#               'Chicken' here; a soil batch might have one entry per location.
#
#  STANDALONE_SAMPLES   samples that get a recovery of their own but belong to
#               no matrix, such as an equipment blank.
#
#  Every Unknown sample must appear exactly once, in one or the other.
#
#  Don't know the sample names? Leave MATRICES empty ({}) and run this cell.
#  It prints the batch's samples and a ready-made block to paste back here.
# ═════════════════════════════════════════════════════════════════════════

MATRICES = {
    'Oyster': ['26_08_04_Oyster_TA_01',     # S Oyster 1
               '26_08_04_Oyster_TA_02',     # S Oyster 2
               '26_08_04_Oyster_TA_03',     # S Oyster 3
               '26_08_04_Oyster_TA_04',     # M Oyster 1
               '26_08_04_Oyster_TA_05',     # M Oyster 2
               '26_08_04_Oyster_TA_06',     # M Oyster 3
               '26_08_04_Oyster_TA_07',     # L Oyster 1
               '26_08_04_Oyster_TA_08',     # L Oyster 2
               '26_08_04_Oyster_TA_09',     # L Oyster 3
               '26_08_04_Oyster_TA_10',     # M Oyster Low Spike
               '26_08_04_Oyster_TA_11'],    # M Oyster High Spike
    'Chicken': ['26_08_04_Oyster_TA_12',    # Chicken Blank 1
                '26_08_04_Oyster_TA_13',    # Chicken Blank 2
                '26_08_04_Oyster_TA_14',    # Chicken Blank 3
                '26_08_04_Oyster_TA_15',    # Chicken Low Spike
                '26_08_04_Oyster_TA_16'],   # Chicken High Spike
}

STANDALONE_SAMPLES = ['26_08_04_Oyster_TA_17']    # Equipment Blank

# ═════════════════════════════════════════════════════════════════════════

if not MATRICES:
    print('Copy the block below into MATRICES above and group the samples.\n')
    print("MATRICES = {\n    'Matrix name': [")
    for name in unknown_samples:
        print(f"        '{name}',    # {sample_id.get(name, '')}")
    print('    ],\n}\n\nSTANDALONE_SAMPLES = []')
    raise ValueError('MATRICES is empty — paste the block above and fill it in.')

assigned = [name for samples in MATRICES.values() for name in samples] + list(STANDALONE_SAMPLES)
problems = [(n, 'not an Unknown sample in this batch') for n in assigned if n not in unknown_samples]
problems += [(n, f'appears {assigned.count(n)} times — each sample belongs to one matrix only')
             for n in sorted(set(assigned)) if assigned.count(n) > 1]
problems += [(n, 'not assigned to any matrix or listed as standalone')
             for n in unknown_samples if n not in assigned]
problems += [(m, 'has no samples') for m, samples in MATRICES.items() if not samples]

if problems:
    print(f'{len(problems)} entr(ies) need attention:\n')
    for where, why in problems:
        print(f'  {where:26s} {str(sample_id.get(where, "")):22s} {why}')
    raise ValueError('Fix the entries listed above in this cell, then re-run it.')

sample_table['matrix'] = [next((m for m, s in MATRICES.items() if name in s), '')
                          for name in sample_table.index]

for matrix, samples in MATRICES.items():
    print(f'{matrix}  ({len(samples)} samples)')
    for name in samples:
        print(f'    {name}    {sample_id.get(name, "")}')
print(f'\nStandalone, recovered on their own ({len(STANDALONE_SAMPLES)}):')
for name in STANDALONE_SAMPLES:
    print(f'    {name}    {sample_id.get(name, "")}')

In [ ]:
# Peak area of every labelled standard, read from its OWN export row.
#
# §7.4 says to take the EIS area from the ISTD Area column on the target rows.
# That loses data: §2 drops a target row whose RT failed, and the EIS area rides
# along with it — 163 of 425 sample/EIS pairs went missing that way, including
# 13 of the equipment blank's 25. The labelled standard has its own export file
# and its own row, which survives independently of any target compound, and
# where both sources exist they agree to within 0.5 counts. The checks assert
# that agreement, so a real divergence would still be caught.
labelled_area = master[master['Compound Name'].isin(LABELLED_STANDARDS)].pivot_table(
    index='Sample Raw File Name', columns='Compound Name', values='Peak Area', aggfunc='mean')
eis_area = labelled_area.reindex(columns=sorted(EIS_COMPOUNDS))

# Cal standards and instrument blanks were never extracted, so the EIS in them is
# the whole amount added. Their mean area is the 100% mark. One per EIS compound.
sample_type = master.drop_duplicates('Sample Raw File Name').set_index(
    'Sample Raw File Name')['Sample Type']
reference_samples = sample_type[sample_type.isin([CAL_STD, INSTRUMENT_BLANK])].index
eis_reference = eis_area.reindex(reference_samples).mean()

method1 = 100 * eis_area.reindex(unknown_samples).div(eis_reference)

eis_table = method1.stack(future_stack=True).rename('recovery_pct').reset_index()
eis_table.columns = ['Sample Raw File Name', 'Compound Name', 'recovery_pct']
eis_table['matrix'] = eis_table['Sample Raw File Name'].map(sample_table['matrix'])
eis_table['sample_id'] = eis_table['Sample Raw File Name'].map(sample_table['sample_id'])

# §7.6 flags, against the EIS window rather than §6's spike window.
recovery = eis_table['recovery_pct']
eis_table['flag'] = ''
eis_table.loc[recovery < EIS_RECOVERY_MIN_PCT, 'flag'] = FLAG_LOW_EIS
eis_table.loc[recovery > EIS_RECOVERY_MAX_PCT, 'flag'] = FLAG_HIGH_EIS
eis_table.loc[recovery.isna(), 'flag'] = FLAG_NO_EIS

# §7.1 reports per matrix, averaging its samples. A standalone sample carries an
# empty matrix, is left out of this, and is shown on its own.
matrix_recovery = eis_table[eis_table['matrix'] != ''].pivot_table(
    index='Compound Name', columns='matrix', values='recovery_pct')

## §7 Method 1 — Checks

What to look at before moving on to Method 2:

- every EIS compound got a recovery in every Unknown sample, and the NIS
  compounds got none, because they are not recovered against
- the EIS list equals the compounds the exports use as an internal standard. The
  cell above raises rather than proceeding if a NIS name is misspelled, which
  matters here because `M3PFBA`/`MPFBA` and `MPFOS`/`M8PFOS` differ by so little
- the reference is the mean EIS area across cal standards and instrument blanks
  only. The check re-derives one by hand from the raw areas
- `ISTD Area` is the same number wherever it is read from for a given sample and
  EIS. The check asserts that, since averaging over target rows would otherwise
  hide a disagreement
- recoveries cluster around 100%. A whole matrix sitting low points at the
  extraction for that matrix; a single compound low across every matrix points at
  the compound
- the equipment blank is reported on its own, not folded into a matrix average
- flags agree with the 50-150% EIS window in both directions. That window is
  wider than §6's 70-130% because an EIS goes through the whole extraction

In [ ]:
assert set(eis_table['Compound Name'].unique()) == EIS_COMPOUNDS, 'an EIS compound is missing'
assert not (set(eis_table['Compound Name'].unique()) & NIS_COMPOUNDS), 'a NIS compound was recovered'
assert len(eis_table) == len(EIS_COMPOUNDS) * len(unknown_samples), 'a sample or compound is missing'
assert (eis_reference > 0).all(), 'an EIS reference area is zero or negative'

# The own-row area must agree with the ISTD Area the exports report on target
# rows, wherever both exist. This is what justifies preferring the own row.
istd_rows = master.dropna(subset=['ISTD Compound Name'])
via_istd = istd_rows.pivot_table(index='Sample Raw File Name', columns='ISTD Compound Name',
                                 values='ISTD Area', aggfunc='mean').reindex(
                                     index=eis_area.index, columns=eis_area.columns)
overlap = eis_area.notna() & via_istd.notna()
assert (eis_area[overlap] - via_istd[overlap]).abs().max().max() <= 1.0, \
    'the EIS own-row area disagrees with the ISTD Area on the target rows'
spread = istd_rows.groupby(['Sample Raw File Name', 'ISTD Compound Name'])['ISTD Area'].nunique()
assert spread.max() == 1, 'ISTD Area disagrees between target rows for the same sample and EIS'

# Re-derive one reference and one recovery by hand from the raw areas.
probe = 'M8PFOS'
by_hand = master[(master['Compound Name'] == probe)
                 & master['Sample Raw File Name'].isin(reference_samples)]['Peak Area'].mean()
assert abs(by_hand - eis_reference[probe]) < 1e-6, f'reference disagrees: {by_hand} vs {eis_reference[probe]}'
probe_sample = unknown_samples[0]
hand = 100 * eis_area.loc[probe_sample, probe] / by_hand
in_table = eis_table.query("`Sample Raw File Name` == @probe_sample and `Compound Name` == @probe")['recovery_pct'].iloc[0]
assert abs(hand - in_table) < 1e-9, f'recovery disagrees: {hand} vs {in_table}'

passed = eis_table['flag'] == ''
assert passed.eq(eis_table['recovery_pct'].between(EIS_RECOVERY_MIN_PCT, EIS_RECOVERY_MAX_PCT)).all(), \
    'a flag disagrees with the recovery window'
print('All checks passed.')
print(f'Missing areas: {int(eis_area.reindex(unknown_samples).isna().values.sum())} of '
      f'{len(EIS_COMPOUNDS) * len(unknown_samples)} sample/EIS pairs '
      f'(reading ISTD Area off target rows instead would lose '
      f'{int(via_istd.reindex(unknown_samples).isna().values.sum())})')

print('\nEIS recovery % by compound and matrix (Method 1)')
print(matrix_recovery.round(1).to_string())

print('\nStandalone samples, reported on their own')
standalone = eis_table[eis_table['matrix'] == '']
print(standalone.pivot_table(index='Compound Name', columns='sample_id',
                             values='recovery_pct').round(1).to_string())

print('\nRecovery spread per sample')
print(eis_table.groupby(['matrix', 'sample_id'])['recovery_pct'].describe()[
    ['count', 'min', '50%', 'max']].round(1).to_string())

flagged = eis_table[eis_table['flag'] != '']
print(f'\nQC FLAGS — EIS recovery outside {EIS_RECOVERY_MIN_PCT:g}-{EIS_RECOVERY_MAX_PCT:g}%: '
      f'{len(flagged)} of {len(eis_table)}')
print('\nFlagged counts by sample')
print(flagged.groupby(['matrix', 'sample_id']).size().to_string())

## §7 — EIS recovery, Method 2

Method 1 asks how much EIS came back compared with standards that were never
extracted. It cannot tell a genuine extraction loss from the instrument simply
injecting less that run, or drifting across a long batch.

Method 2 removes that doubt using the NIS. The NIS goes in **after** extraction,
so it never had the chance to be lost. Anything that changed the NIS signal —
injection volume, source response, drift — changed the EIS signal the same way,
so comparing EIS against its own NIS divides out everything but the extraction.

It runs in two steps. First, how much EIS the vial actually contained:

    response factor = mean over cal standards of (EIS area / NIS area)

    EIS concentration = (EIS area / NIS area in the sample)
                        / response factor x theoretical concentration
                        x dilution factor

Then that measured concentration against the amount that should have been there:

    recovery % = EIS concentration / theoretical concentration x 100

Both are in **ng/L, in the vial**. No sample weight enters. The same amount of
EIS goes into every sample whatever it weighed, so its recovery must not depend
on the weighing — dividing by sample weight would make a heavier sample look
like a worse extraction.

**The theoretical concentration is per compound**, read from `ISTD Amount`: 18 of
the 25 EIS compounds carry 5000 ng/L and seven do not, down to 4660 for M3PFBS.

**Only seven compounds get a Method 2 value**, the ones with a properly matching
NIS. The other 18 keep their Method 1 value and are not flagged here.

**The QC window is 50-150%**, the same as Method 1 and wider than §6's 70-130%
for spike recovery.

**A note on the mass terms.** §7.5 builds its response factor from `Detected
Mass`. That column holds the same value in every sample — 506.96 for M8PFOS,
498.93 for PFOS, unchanged across a calibration curve spanning 10 to 50000 ng/L —
so it is an m/z, not an amount, and cannot serve as a mass. Carried through the
algebra the NIS mass cancels, which is why no NIS amount is needed anywhere.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  DILUTION FACTOR FOR THIS BATCH — type the value here.
#
#  §7.5 applies a dilution factor to the Method 2 result. Leave it at 1 unless
#  this batch was diluted before injection, in which case enter the factor.
# ══════════════════════════════════════════════════════════════════════════

DILUTION_FACTOR = 1.0

# ══════════════════════════════════════════════════════════════════════════
#  NIS TO EIS PAIRING — from the method, not from this batch.
#
#  Which NIS corrects which EIS. Only a pair that matches properly is listed;
#  an EIS with no matching NIS gets no Method 2 value and is not flagged for it.
#  Read each line as "this EIS is corrected by this NIS". Note how close some
#  names are — MPFBA is corrected by M3PFBA, M8PFOS by MPFOS.
# ══════════════════════════════════════════════════════════════════════════

NIS_FOR_EIS = {
    'MPFBA':   'M3PFBA',
    'M5PFHxA': 'MPFHxA',
    'M8PFOA':  'MPFOA',
    'M9PFNA':  'MPFNA',
    'M6PFDA':  'MPFDA',
    'M3PFHxS': 'MPFHxS',
    'M8PFOS':  'MPFOS',
}

# ══════════════════════════════════════════════════════════════════════════

if not DILUTION_FACTOR > 0:
    raise ValueError(f'DILUTION_FACTOR must be greater than zero, not {DILUTION_FACTOR!r}')

bad_eis = sorted(set(NIS_FOR_EIS) - EIS_COMPOUNDS)
bad_nis = sorted(set(NIS_FOR_EIS.values()) - NIS_COMPOUNDS)
if bad_eis or bad_nis:
    raise KeyError(f'Pairing does not match this batch. Not an EIS: {bad_eis}. '
                   f'Not a NIS: {bad_nis}. Check for an EIS and NIS name swapped round.')

print(f'Dilution factor: {DILUTION_FACTOR:g}')
print(f'\n{len(NIS_FOR_EIS)} of {len(EIS_COMPOUNDS)} EIS compounds have a matching NIS:')
for eis, nis in NIS_FOR_EIS.items():
    print(f'    {eis:10s} corrected by  {nis}')
print(f'\nThe other {len(EIS_COMPOUNDS) - len(NIS_FOR_EIS)} keep their Method 1 value only.')

In [ ]:
# The NIS went in after extraction, so its area carries the instrument's
# behaviour and nothing about the extraction. Dividing EIS by NIS cancels it.
nis_area = labelled_area.reindex(columns=sorted(NIS_COMPOUNDS))

paired_eis = sorted(NIS_FOR_EIS)
ratio = pd.DataFrame({eis: eis_area[eis] / nis_area[NIS_FOR_EIS[eis]] for eis in paired_eis})

# §7.5 builds the response factor in the cal standards, so the reference comes
# from those alone — not the instrument blanks Method 1 also uses.
cal_samples = sample_type[sample_type == CAL_STD].index
ratio_reference = ratio.reindex(cal_samples).mean()

# The theoretical concentration of each EIS, in ng/L, as the exports report it.
# Constant per compound across the batch, since the same amount goes into every
# sample. The checks assert that before it is relied on.
EIS_THEORETICAL = master.dropna(subset=['ISTD Compound Name']).drop_duplicates(
    'ISTD Compound Name').set_index('ISTD Compound Name')['ISTD Amount']
theoretical = EIS_THEORETICAL.reindex(paired_eis)

# Step 1 — how much EIS the vial actually held, in ng/L. A cal standard holds the
# EIS at its theoretical concentration by definition, so a sample whose EIS/NIS
# ratio matches the standards read back the full amount.
eis_concentration = (DILUTION_FACTOR
                     * ratio.reindex(unknown_samples).div(ratio_reference)
                     * theoretical)

# Step 2 — that against what should have been there. Both sides are ng/L in the
# vial, so no sample weight enters.
method2 = 100 * eis_concentration.div(theoretical)

method2_table = method2.stack(future_stack=True).rename('recovery_pct').reset_index()
method2_table.columns = ['Sample Raw File Name', 'Compound Name', 'recovery_pct']
method2_table['eis_ng_per_L'] = eis_concentration.stack(future_stack=True).values
method2_table['theoretical_ng_per_L'] = method2_table['Compound Name'].map(theoretical)
method2_table['matrix'] = method2_table['Sample Raw File Name'].map(sample_table['matrix'])
method2_table['sample_id'] = method2_table['Sample Raw File Name'].map(sample_table['sample_id'])

m2recovery = method2_table['recovery_pct']
method2_table['flag'] = ''
method2_table.loc[m2recovery < EIS_RECOVERY_MIN_PCT, 'flag'] = FLAG_LOW_EIS
method2_table.loc[m2recovery > EIS_RECOVERY_MAX_PCT, 'flag'] = FLAG_HIGH_EIS
method2_table.loc[m2recovery.isna(), 'flag'] = FLAG_NO_EIS

matrix_recovery2 = method2_table[method2_table['matrix'] != ''].pivot_table(
    index='Compound Name', columns='matrix', values='recovery_pct')

## §7 Method 2 — Checks

What to look at:

- only the seven paired compounds appear, and every one of them got a value in
  every sample. The other 18 EIS compounds are absent rather than zero
- the pairing cell refuses to run if an EIS and a NIS have been swapped round,
  which matters because `MPFBA`/`M3PFBA` and `MPFOS`/`M8PFOS` differ so little
- the reference ratio comes from the cal standards only, as §7.5 specifies,
  where Method 1 pools cal standards with instrument blanks
- the check re-derives one recovery by hand from four raw peak areas
- compare Method 2 against Method 1 for the same compound and sample. Where they
  agree, the loss is real extraction loss. Where Method 2 is much higher, the
  instrument was injecting or responding low and Method 1 overstated the loss
- the NIS areas themselves should be similar across samples. A NIS area that
  swings wildly means the NIS is not doing its job and Method 2 inherits that

In [ ]:
assert set(method2_table['Compound Name'].unique()) == set(NIS_FOR_EIS), 'a paired compound is missing'
assert len(method2_table) == len(NIS_FOR_EIS) * len(unknown_samples), 'a sample or compound is missing'
assert (ratio_reference > 0).all(), 'a reference ratio is zero or negative'
assert len(cal_samples) > 0, 'no cal standards to build the reference from'

# The theoretical concentration must be one value per EIS across the whole batch.
per_compound = master.dropna(subset=['ISTD Compound Name']).groupby(
    'ISTD Compound Name')['ISTD Amount'].nunique()
assert per_compound.max() == 1, 'an EIS carries more than one theoretical concentration'
assert theoretical.notna().all(), 'an EIS has no theoretical concentration'

# Re-derive one concentration and recovery by hand from four raw peak areas.
pe, pn, ps = 'M8PFOS', NIS_FOR_EIS['M8PFOS'], unknown_samples[0]
area_of = lambda c, s: master[(master['Compound Name'] == c)
                              & (master['Sample Raw File Name'] == s)]['Peak Area'].mean()
sample_ratio = area_of(pe, ps) / area_of(pn, ps)
cal_ratio = pd.Series({s: area_of(pe, s) / area_of(pn, s) for s in cal_samples}).mean()
hand_conc = DILUTION_FACTOR * sample_ratio / cal_ratio * theoretical[pe]
assert abs(hand_conc - eis_concentration.loc[ps, pe]) < 1e-6, 'concentration disagrees by hand'
hand = 100 * hand_conc / theoretical[pe]
in_table = method2_table.query("`Sample Raw File Name` == @ps and `Compound Name` == @pe")['recovery_pct'].iloc[0]
assert abs(hand - in_table) < 1e-6, f'Method 2 disagrees by hand: {hand} vs {in_table}'

passed2 = method2_table['flag'] == ''
assert passed2.eq(method2_table['recovery_pct'].between(EIS_RECOVERY_MIN_PCT, EIS_RECOVERY_MAX_PCT)).all(), \
    'a flag disagrees with the recovery window'
print('All checks passed.')

print('\nTheoretical EIS concentration used (ng/L)')
print(theoretical.to_string())

print('\nNIS peak area across all samples — a NIS should be steady')
print(nis_area.describe().loc[['min', '50%', 'max']].round(0).to_string())

print('\nMeasured EIS concentration (ng/L), by compound and matrix')
print(method2_table[method2_table['matrix'] != ''].pivot_table(
    index='Compound Name', columns='matrix', values='eis_ng_per_L').round(0).to_string())

print('\nEIS recovery % by compound and matrix (Method 2)')
print(matrix_recovery2.round(1).to_string())

print('\nMethod 1 against Method 2, per matrix')
compare = matrix_recovery.loc[paired_eis].join(matrix_recovery2, lsuffix=' M1', rsuffix=' M2')
print(compare[sorted(compare.columns)].round(1).to_string())

print('\nStandalone samples (Method 2)')
print(method2_table[method2_table['matrix'] == ''].pivot_table(
    index='Compound Name', columns='sample_id', values='recovery_pct').round(1).to_string())

flagged2 = method2_table[method2_table['flag'] != '']
print(f'\nQC FLAGS — Method 2 outside {EIS_RECOVERY_MIN_PCT:g}-{EIS_RECOVERY_MAX_PCT:g}%: '
      f'{len(flagged2)} of {len(method2_table)}')
print('\nFlagged counts by sample')
print(flagged2.groupby(['matrix', 'sample_id']).size().to_string())